# Toolsmith: canonical Colab notebook

This is the single notebook for the Toolsmith track part. Run its sections in order on a Colab T4 or L4 runtime: environment setup, raw serving, completion smoke test, and the Day 2 prerequisite tool-calling gate. Later Toolsmith milestones extend this same notebook. The repository is synchronized with `git pull --ff-only`; the notebook never resets or discards checkout changes. Put `HF_TOKEN` and `LOCAL_KEY` in `MyDrive/toolsmith/.env` (or `.env.txt`), or configure them in Colab Secrets.

llama.cpp is cached as a small runtime containing only the server and shared libraries. The GGUF is stored as a plain Drive file and copied to local disk with exact-size validation and progress output. Record Colab units before and after the session; the notebook cannot read them automatically.

In [ ]:
import json
import os
import platform
import shlex
import shutil
import socket
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    check=True,
    capture_output=True,
    text=True,
)
gpu_info = gpu_result.stdout.strip()
print(gpu_info)
print(f'Python {platform.python_version()}')

In [ ]:
from google.colab import drive, userdata

REPO_URL = 'https://github.com/shreyasnarahari/bare-metal-agent'
PROJECT_DIR = Path('/content/bare-metal-agent')
if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main'],
        check=True,
    )
elif PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} exists but is not a bare-metal-agent git checkout.'
    )
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / 'pyproject.toml').is_file():
    raise FileNotFoundError(f'Expected {PROJECT_DIR / "pyproject.toml"} after repository setup.')
src_dir = PROJECT_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/toolsmith')
DRIVE_BUILD_CACHE = DRIVE_ROOT / 'builds'
DRIVE_MODEL_CACHE = DRIVE_ROOT / 'models'
DRIVE_RECORDS = DRIVE_ROOT / 'records'
for directory in (DRIVE_BUILD_CACHE, DRIVE_MODEL_CACHE, DRIVE_RECORDS):
    directory.mkdir(parents=True, exist_ok=True)

def read_env_file(path):
    values = {}
    if not path.is_file():
        return values
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        key, separator, value = line.partition('=')
        if not separator:
            continue
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in '"\'':
            value = value[1:-1]
        values[key.strip()] = value
    return values

env_path = next(
    (
        candidate
        for candidate in (DRIVE_ROOT / '.env', DRIVE_ROOT / '.env.txt')
        if candidate.is_file()
    ),
    DRIVE_ROOT / '.env',
)
env_values = read_env_file(env_path)
def load_secret(name):
    value = env_values.get(name, '').strip()
    if value:
        return value, env_path
    return userdata.get(name), 'Colab Secrets'

HF_TOKEN, hf_source = load_secret('HF_TOKEN')
LOCAL_KEY, local_source = load_secret('LOCAL_KEY')
if not HF_TOKEN or not LOCAL_KEY:
    raise ValueError(f'HF_TOKEN and LOCAL_KEY must be set in {env_path} or Colab Secrets.')
print(f'HF_TOKEN loaded from {hf_source}; LOCAL_KEY loaded from {local_source}')

In [ ]:
get_ipython().run_line_magic(
    'pip',
    f"install -q -e {shlex.quote(str(PROJECT_DIR) + '[colab]')}",
)

## Build or restore llama.cpp

Every local or cached executable is checked with `--version` and its runtime library path before use. A failed or interrupted restore cannot be mistaken for a complete build. The Drive cache uses a completion marker and is replaced only after all runtime files have been copied.

In [ ]:
LLAMA_CPP_REF = 'b5401'
LOCAL_SOURCE = Path('/content/llama.cpp-source')
LOCAL_BUILD = Path('/content/llama.cpp-build')
def query_gpu(field):
    result = subprocess.run(
        ['nvidia-smi', f'--query-gpu={field}', '--format=csv,noheader'],
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip().splitlines()[0].strip()

GPU_NAME = query_gpu('name')
GPU_COMPUTE_CAP = query_gpu('compute_cap').replace('.', '')
CACHED_BUILD = DRIVE_BUILD_CACHE / f'llama.cpp-{LLAMA_CPP_REF}-cuda-sm{GPU_COMPUTE_CAP}-runtime-v2'
SERVER_BIN = LOCAL_BUILD / 'bin' / 'llama-server'
print(
    f'{GPU_NAME} compute capability {GPU_COMPUTE_CAP}; cache {CACHED_BUILD}',
    flush=True,
)

def remove_directory(path):
    if path.exists():
        shutil.rmtree(path)

def server_environment(root):
    environment = os.environ.copy()
    paths = [str(root / 'lib'), str(root / 'bin')]
    existing = environment.get('LD_LIBRARY_PATH')
    if existing:
        paths.append(existing)
    environment['LD_LIBRARY_PATH'] = os.pathsep.join(paths)
    return environment

def server_works(root):
    binary = root / 'bin' / 'llama-server'
    if not binary.is_file():
        return False
    try:
        binary.chmod(binary.stat().st_mode | 0o755)
        result = subprocess.run(
            [str(binary), '--version'],
            cwd=str(binary.parent),
            env=server_environment(root),
            capture_output=True,
            text=True,
            timeout=120,
        )
    except (OSError, subprocess.TimeoutExpired) as error:
        print(f'Server validation failed for {binary}: {error}')
        return False
    if result.returncode != 0:
        print(f'Server validation failed for {binary}: {result.stderr.strip()}')
        return False
    print(result.stdout.strip() or result.stderr.strip())
    return True

def restore_cached_build():
    def list_directory(path):
        try:
            return list(path.iterdir())
        except FileNotFoundError:
            return []
        except OSError as error:
            print(f'Could not list {path}: {error}', flush=True)
            return []

    def usable_cache(candidate):
        # A fresh Drive mount can hide children from stat until the directory is listed.
        list_directory(candidate)
        list_directory(candidate / 'bin')
        marker = candidate / '.complete'
        if marker.is_file() and marker.read_text() != LLAMA_CPP_REF + '\n':
            print(f'Ignoring {candidate}: completion marker does not match {LLAMA_CPP_REF}', flush=True)
            return False
        binary = candidate / 'bin' / 'llama-server'
        if not binary.is_file():
            print(f'No cached llama-server at {binary}', flush=True)
            return False
        expected = globals().get('GPU_COMPUTE_CAP')
        if expected:
            cap_marker = candidate / '.compute-cap'
            recorded = cap_marker.read_text().strip() if cap_marker.is_file() else ''
            if recorded != expected:
                print(
                    f'Ignoring {candidate}: CUDA architecture {recorded or "unknown"} '
                    f'does not match this GPU ({expected}).',
                    flush=True,
                )
                return False
        return True

    def copy_cached_runtime(source, destination):
        # The previous notebook stored the whole CMake tree without a marker.
        if (source / 'CMakeCache.txt').is_file() and not (source / '.complete').is_file():
            (destination / 'bin').mkdir(parents=True)
            shutil.copy2(source / 'bin' / 'llama-server', destination / 'bin' / 'llama-server')
            for folder_name in ('bin', 'lib'):
                folder = source / folder_name
                if not folder.is_dir():
                    continue
                target_folder = destination / folder_name
                target_folder.mkdir(parents=True, exist_ok=True)
                for artifact in folder.iterdir():
                    if artifact.name == 'llama-server':
                        continue
                    if not (artifact.name.startswith('lib') and '.so' in artifact.name):
                        continue
                    target = target_folder / artifact.name
                    if artifact.is_symlink():
                        target.symlink_to(os.readlink(artifact))
                    else:
                        shutil.copy2(artifact, target)
            return
        shutil.copytree(source, destination, symlinks=True)

    list_directory(CACHED_BUILD.parent)
    if not usable_cache(CACHED_BUILD):
        return False
    source = CACHED_BUILD
    staging = Path(os.environ.get('TOOLSMITH_RUNTIME_STAGING', '/content/llama.cpp-runtime-staging'))
    remove_directory(staging)
    print(f'Restoring runtime from {source}', flush=True)
    copy_cached_runtime(source, staging)
    if not server_works(staging):
        remove_directory(staging)
        print('Cached runtime is invalid; rebuilding locally.')
        return False
    remove_directory(LOCAL_BUILD)
    staging.rename(LOCAL_BUILD)
    return True

def cache_local_build():
    staging = CACHED_BUILD.with_name(CACHED_BUILD.name + '-staging')
    remove_directory(staging)
    (staging / 'bin').mkdir(parents=True)
    shutil.copy2(SERVER_BIN, staging / 'bin' / 'llama-server')
    for artifact in (LOCAL_BUILD / 'bin').iterdir():
        if artifact.name.startswith('lib') and '.so' in artifact.name:
            if artifact.is_symlink():
                target = os.readlink(artifact)
                (staging / 'bin' / artifact.name).symlink_to(target)
            else:
                shutil.copy2(artifact, staging / 'bin' / artifact.name)
    if (LOCAL_BUILD / 'lib').is_dir():
        shutil.copytree(LOCAL_BUILD / 'lib', staging / 'lib', symlinks=True)
    cap = globals().get('GPU_COMPUTE_CAP')
    if cap:
        (LOCAL_BUILD / '.compute-cap').write_text(cap + '\n')
        (staging / '.compute-cap').write_text(cap + '\n')
    (staging / '.complete').write_text(LLAMA_CPP_REF + '\n')
    remove_directory(CACHED_BUILD)
    staging.rename(CACHED_BUILD)
    print(f'Cached validated runtime at {CACHED_BUILD}')

def runtime_matches_gpu(root):
    marker = root / '.compute-cap'
    recorded = marker.read_text().strip() if marker.is_file() else ''
    if recorded == GPU_COMPUTE_CAP:
        return True
    print(
        f'{root} was built for CUDA architecture {recorded or "unknown"}, '
        f'but this GPU is {GPU_COMPUTE_CAP}.',
        flush=True,
    )
    return False

if server_works(LOCAL_BUILD) and runtime_matches_gpu(LOCAL_BUILD):
    print(f'Using validated local build: {SERVER_BIN}')
elif not restore_cached_build():
    remove_directory(LOCAL_BUILD)
    if not (LOCAL_SOURCE / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--filter=blob:none', 'https://github.com/ggml-org/llama.cpp.git', str(LOCAL_SOURCE)],
            check=True,
        )
    subprocess.run(
        ['git', '-C', str(LOCAL_SOURCE), 'fetch', '--depth', '1', 'origin', 'tag', LLAMA_CPP_REF],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(LOCAL_SOURCE), 'checkout', '--detach', LLAMA_CPP_REF],
        check=True,
    )
    subprocess.run(
        [
            'cmake', '-S', str(LOCAL_SOURCE), '-B', str(LOCAL_BUILD),
            '-DGGML_CUDA=ON', f'-DCMAKE_CUDA_ARCHITECTURES={GPU_COMPUTE_CAP}',
            '-DCMAKE_BUILD_TYPE=Release', '-DLLAMA_CURL=ON',
        ],
        check=True,
    )
    subprocess.run(
        ['cmake', '--build', str(LOCAL_BUILD), '--target', 'llama-server', '-j', '8'],
        check=True,
    )
    if not server_works(LOCAL_BUILD):
        raise RuntimeError('The local build finished but llama-server failed runtime validation.')
    cache_local_build()

os.environ['LD_LIBRARY_PATH'] = server_environment(LOCAL_BUILD)['LD_LIBRARY_PATH']
if not (server_works(LOCAL_BUILD) and runtime_matches_gpu(LOCAL_BUILD)):
    raise RuntimeError(f'llama-server is not usable for this GPU at {SERVER_BIN}')
print(f'Using llama.cpp {LLAMA_CPP_REF} for {GPU_NAME} (sm_{GPU_COMPUTE_CAP}): {SERVER_BIN}')

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

from toolsmith.artifacts import copy_file_atomic, is_gguf

MODEL_REPO = 'Qwen/Qwen3-8B-GGUF'
MODEL_FILE = 'Qwen3-8B-Q4_K_M.gguf'
LOCAL_MODEL = Path('/content') / MODEL_FILE
DRIVE_MODEL_FILE = DRIVE_MODEL_CACHE / MODEL_FILE
model_info = HfApi(token=HF_TOKEN).model_info(MODEL_REPO, files_metadata=True)
MODEL_REVISION = model_info.sha
file_metadata = next(
    (entry for entry in model_info.siblings if entry.rfilename == MODEL_FILE),
    None,
)
if file_metadata is None or file_metadata.size is None:
    raise RuntimeError(f'Could not determine the expected size of {MODEL_FILE}')
EXPECTED_MODEL_SIZE = file_metadata.size

def copy_with_progress(source, destination):
    state = {'last_step': -1}
    def report(copied, total):
        percent = int(copied * 100 / total) if total else 100
        step = percent // 20
        if step != state['last_step']:
            print(f'{destination.name}: {percent}% ({copied:,}/{total:,} bytes)', flush=True)
            state['last_step'] = step
    return copy_file_atomic(
        source,
        destination,
        expected_size=EXPECTED_MODEL_SIZE,
        progress=report,
    )

if not is_gguf(LOCAL_MODEL, EXPECTED_MODEL_SIZE):
    if is_gguf(DRIVE_MODEL_FILE, EXPECTED_MODEL_SIZE):
        print(f'Copying cached GGUF from {DRIVE_MODEL_FILE}', flush=True)
        copy_with_progress(DRIVE_MODEL_FILE, LOCAL_MODEL)
    else:
        print(f'Downloading {MODEL_FILE} at revision {MODEL_REVISION}', flush=True)
        downloaded = Path(
            hf_hub_download(
                repo_id=MODEL_REPO,
                filename=MODEL_FILE,
                revision=MODEL_REVISION,
                cache_dir='/content/hf-cache',
                token=HF_TOKEN,
            )
        )
        if not is_gguf(downloaded, EXPECTED_MODEL_SIZE):
            downloaded = Path(
                hf_hub_download(
                    repo_id=MODEL_REPO,
                    filename=MODEL_FILE,
                    revision=MODEL_REVISION,
                    cache_dir='/content/hf-cache',
                    token=HF_TOKEN,
                    force_download=True,
                )
            )
        if not is_gguf(downloaded, EXPECTED_MODEL_SIZE):
            raise RuntimeError(f'Downloaded model failed GGUF validation: {downloaded}')
        copy_with_progress(downloaded, LOCAL_MODEL)

if not is_gguf(DRIVE_MODEL_FILE, EXPECTED_MODEL_SIZE):
    print(f'Caching validated GGUF in {DRIVE_MODEL_FILE}', flush=True)
    copy_with_progress(LOCAL_MODEL, DRIVE_MODEL_FILE)
if not is_gguf(LOCAL_MODEL, EXPECTED_MODEL_SIZE):
    raise RuntimeError(f'Local GGUF validation failed: {LOCAL_MODEL}')
print(f'Local model: {LOCAL_MODEL} ({EXPECTED_MODEL_SIZE:,} bytes)')
print(f'Model revision: {MODEL_REVISION}')

In [ ]:
from toolsmith.artifacts import is_gguf
from toolsmith.backends import (
    CompletionTruncatedError,
    LlamaCppClient,
    LlamaCppServer,
)
from toolsmith.recording import JsonlRecorder

required = [
    'SERVER_BIN', 'LOCAL_MODEL', 'EXPECTED_MODEL_SIZE', 'LOCAL_KEY',
    'HF_TOKEN', 'DRIVE_RECORDS', 'LLAMA_CPP_REF', 'MODEL_REVISION',
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Run the earlier cells first; missing state: {missing}')
if not is_gguf(LOCAL_MODEL, EXPECTED_MODEL_SIZE):
    raise RuntimeError(f'GGUF validation failed immediately before startup: {LOCAL_MODEL}')

previous = globals().get('server')
if previous is not None and previous.is_running:
    previous.stop(timeout=10)

def first_free_port(host, preferred=8080, span=20):
    for port in range(preferred, preferred + span):
        with socket.socket() as probe:
            try:
                probe.bind((host, port))
            except OSError:
                continue
            return port
    raise RuntimeError(
        f'No free port on {host} from {preferred} through {preferred + span - 1}. '
        'No process was killed.'
    )

SERVER_PORT = first_free_port('127.0.0.1')
if SERVER_PORT != 8080:
    print(f'Port 8080 is in use; llama-server will listen on {SERVER_PORT}', flush=True)

SERVER_LOG_PATH = Path('/content/llama-server.log')
SERVER_LOG_PATH.unlink(missing_ok=True)
SERVER_BIN.chmod(SERVER_BIN.stat().st_mode | 0o755)
server = LlamaCppServer(
    executable=SERVER_BIN,
    model=LOCAL_MODEL,
    api_key=LOCAL_KEY,
    log_path=SERVER_LOG_PATH,
    port=SERVER_PORT,
    context_size=16384,
    gpu_layers=99,
)
server.start()
print(f'Server PID: {server.process.pid}; log: {SERVER_LOG_PATH}')

In [ ]:
client = LlamaCppClient(server.base_url, LOCAL_KEY, timeout=10.0)
health = server.wait_until_healthy(client, timeout=600, poll_interval=2)
print(json.dumps(health, indent=2))

## Raw completion smoke test

This request intentionally uses `/completion`, not llama.cpp's chat endpoint. Toolsmith owns prompt rendering and tool-call parsing in later milestones.

In [ ]:
prompt = 'Name one primary color. Respond with one word only.\nAnswer:'
SEED = 0
started = time.perf_counter()
try:
    response = client.complete(
        prompt,
        max_tokens=128,
        temperature=0.0,
        seed=SEED,
    )
    outcome = 'ok'
except CompletionTruncatedError as error:
    response = error.response
    outcome = 'truncated'
elapsed_seconds = time.perf_counter() - started

record = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'llama_cpp_ref': LLAMA_CPP_REF,
    'model_repo': MODEL_REPO,
    'model_file': MODEL_FILE,
    'model_revision': MODEL_REVISION,
    'gpu': gpu_info,
    'python_version': platform.python_version(),
    'server': server.settings(),
    'request': {
        'prompt': prompt,
        'max_tokens': 128,
        'temperature': 0.0,
        'seed': SEED,
        'stream': False,
    },
    'response': response,
    'outcome': outcome,
    'elapsed_seconds': elapsed_seconds,
}
record_path = DRIVE_RECORDS / 'evening-01-smoke.jsonl'
JsonlRecorder(record_path, secrets=(HF_TOKEN, LOCAL_KEY)).append(record)

print(json.dumps(response, indent=2))
print(f'Elapsed: {elapsed_seconds:.2f}s; record: {record_path}')
if outcome != 'ok':
    raise RuntimeError('The smoke test was truncated; inspect the recorded response before continuing.')

## Day 2 prerequisite gate: native tool calling

This section starts our FastAPI shim on port 8000 and uses the official OpenAI client. It verifies ordinary text, a Qwen3 tool call parsed by our code, and a second request containing the assistant call plus a matching tool result. It reuses the server, secrets, installation, and Drive records from the sections above. The agent state machine is intentionally separate from this prerequisite gate.

In [ ]:
import threading
from typing import Literal

import uvicorn
from openai import OpenAI

from toolsmith.api import ToolsmithService, create_app
from toolsmith.prompting import ChatPromptRenderer
from toolsmith.registry import ToolRegistry

if 'shim_server' in globals():
    shim_server.should_exit = True
    if 'shim_thread' in globals():
        shim_thread.join(timeout=10)
if 'openai_client' in globals():
    openai_client.close()

def close_listening_fds(port):
    closed = 0
    for entry in Path('/proc/self/fd').iterdir():
        try:
            if not os.readlink(entry).startswith('socket:'):
                continue
            fd = int(entry.name)
            held = socket.fromfd(fd, socket.AF_INET, socket.SOCK_STREAM)
        except (OSError, ValueError):
            continue
        try:
            bound = held.getsockname()
            if not isinstance(bound, tuple) or len(bound) < 2:
                held.close()
                continue
            bound_port = bound[1]
            accepting = held.getsockopt(socket.SOL_SOCKET, socket.SO_ACCEPTCONN)
        except OSError:
            held.close()
            continue
        held.close()
        if bound_port == port and accepting:
            os.close(fd)
            closed += 1
    return closed

def first_free_port(host, preferred=8080, span=20):
    for port in range(preferred, preferred + span):
        with socket.socket() as probe:
            try:
                probe.bind((host, port))
            except OSError:
                continue
            return port
    raise RuntimeError(
        f'No free port on {host} from {preferred} through {preferred + span - 1}.'
    )

close_listening_fds(8000)
SHIM_PORT = first_free_port('127.0.0.1', 8000)
if SHIM_PORT != 8000:
    print(f'Port 8000 is in use; FastAPI shim will listen on {SHIM_PORT}', flush=True)

renderer = ChatPromptRenderer.from_pretrained('Qwen/Qwen3-8B', token=HF_TOKEN)
recorder = JsonlRecorder(
    DRIVE_RECORDS / 'day2-m0-gate.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
service = ToolsmithService(client, renderer, recorder=recorder)
app = create_app(service, api_key=LOCAL_KEY)
shim_server = uvicorn.Server(
    uvicorn.Config(
        app, host='127.0.0.1', port=SHIM_PORT, log_level='warning',
        timeout_graceful_shutdown=5,
    )
)
shim_thread = threading.Thread(target=shim_server.run, daemon=True)
shim_thread.start()
deadline = time.monotonic() + 30
while not shim_server.started and time.monotonic() < deadline:
    if not shim_thread.is_alive():
        raise RuntimeError('FastAPI shim stopped during startup')
    time.sleep(0.1)
if not shim_server.started:
    raise TimeoutError(f'FastAPI shim did not start on port {SHIM_PORT}')

openai_client = OpenAI(
    base_url=f'http://127.0.0.1:{SHIM_PORT}/v1',
    api_key=LOCAL_KEY,
    timeout=180.0,
    max_retries=0,
)
MODEL_NAME = 'Qwen/Qwen3-8B'

In [ ]:
text_response = openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': 'Name one primary color. Respond with one word.'}],
    max_tokens=128,
    temperature=0.0,
    seed=0,
    extra_body={'chat_template_kwargs': {'enable_thinking': False}},
)
text_answer = text_response.choices[0].message.content
if not text_answer:
    raise AssertionError('The no-tool gate returned an empty answer')
print('No-tool response:', text_answer)

In [ ]:
def get_weather(city: str, unit: Literal['c', 'f'] = 'c') -> dict:
    """Get deterministic demonstration weather for a city."""
    return {'city': city, 'temperature': 22, 'unit': unit, 'condition': 'sunny'}

registry = ToolRegistry()
registry.function(get_weather)
messages = [
    {
        'role': 'user',
        'content': 'What is the current weather in Paris? Use get_weather with city Paris.',
    }
]
generation = {
    'max_tokens': 256,
    'temperature': 0.0,
    'seed': 0,
    'extra_body': {'chat_template_kwargs': {'enable_thinking': False}},
}
first = openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=registry.to_openai(),
    tool_choice='auto',
    **generation,
)
assistant = first.choices[0].message
if not assistant.tool_calls or len(assistant.tool_calls) != 1:
    raise AssertionError(f'Expected exactly one tool call, got: {assistant}')
call = assistant.tool_calls[0]
if call.function.name != 'get_weather':
    raise AssertionError(f'Wrong tool selected: {call.function.name}')
arguments = json.loads(call.function.arguments)
tool_result = get_weather(**arguments)
messages.extend(
    [
        assistant.model_dump(exclude_none=True),
        {
            'role': 'tool',
            'tool_call_id': call.id,
            'content': json.dumps(tool_result),
        },
    ]
)
second = openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=registry.to_openai(),
    tool_choice='auto',
    **generation,
)
final_answer = second.choices[0].message.content
if not final_answer or '22' not in final_answer:
    raise AssertionError(f'Final answer did not incorporate the tool result: {final_answer}')
print('Parsed arguments:', arguments)
print('Tool result:', tool_result)
print('Final response:', final_answer)
print('M0 gate passed; raw records:', DRIVE_RECORDS / 'day2-m0-gate.jsonl')

## Days 3–4: non-streaming and streaming parsers

These cells replay adversarial protocol examples with arbitrary chunks, compare raw llama.cpp streaming and non-streaming output for the same request, and consume our OpenAI-compatible stream with the official SDK. All checks reuse this notebook's existing server, shim, and cleanup.

In [ ]:
import random

from toolsmith.parsing import Qwen3IncrementalParser, Qwen3ToolCallParser

def id_source():
    return (f'call_{index}' for index in range(10))

def replay_sample(text, expect_thinking, chunks):
    ids = id_source()
    parser = Qwen3IncrementalParser(
        lambda: next(ids),
        expect_thinking=expect_thinking,
    )
    timeline = []
    for chunk in chunks:
        for event in parser.feed(chunk):
            timeline.append(type(event).__name__)
    for event in parser.finish():
        timeline.append(type(event).__name__)
    return parser.response, timeline

samples = [
    ('plain', 'plain café text', None),
    (
        'nested',
        '<tool_call>{"name":"weather","arguments":{"city":"Paris","note":"</tool_call>"}}</tool_call>',
        False,
    ),
    (
        'parallel',
        '<think></think>Before<tool_call>{"name":"a","arguments":{}}</tool_call>Middle<tool_call>{"name":"b","arguments":{"x":1}}</tool_call>After',
        True,
    ),
]
fixture_records = []
for name, text, expect_thinking in samples:
    ids = id_source()
    expected = Qwen3ToolCallParser(lambda: next(ids)).parse(text)
    chunk_sets = [
        list(text),
        [text[index:index + 7] for index in range(0, len(text), 7)],
    ]
    randomizer = random.Random(17)
    randomized = []
    remaining = text
    while remaining:
        size = randomizer.randint(1, 9)
        randomized.append(remaining[:size])
        remaining = remaining[size:]
    chunk_sets.append(randomized)
    for chunks in chunk_sets:
        response, timeline = replay_sample(text, expect_thinking, chunks)
        if response != expected:
            raise AssertionError(f'Chunk-invariance failed for {name}: {response} != {expected}')
    print(f'{name}: {len(timeline)} events; first events: {timeline[:12]}')
    fixture_records.append(
        {
            'name': name,
            'raw_text': text,
            'expect_thinking': expect_thinking,
            'content': expected.content,
            'reasoning_content': expected.reasoning_content,
            'tool_calls': [call.to_openai() for call in expected.tool_calls],
        }
    )
JsonlRecorder(
    DRIVE_RECORDS / 'days-03-04-parser-fixtures.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append({'event': 'parser_fixture_replay', 'fixtures': fixture_records})
print('Offline chunk-invariance checks passed')

In [ ]:
stream_messages = [
    {
        'role': 'user',
        'content': 'What is the current weather in Paris? Use get_weather with city Paris.',
    }
]
stream_prompt = renderer.render(
    stream_messages,
    registry.to_openai(),
    enable_thinking=False,
)
raw_settings = {
        'max_tokens': 256,
        'temperature': 0.0,
        'seed': 0,
}
raw_non_stream = client.complete(stream_prompt, **raw_settings)
raw_stream_events = list(client.stream_complete(stream_prompt, **raw_settings))
raw_stream_text = ''.join(event['content'] for event in raw_stream_events)
if raw_stream_text != raw_non_stream['content']:
    raise AssertionError(
        f'Raw stream/non-stream mismatch: {raw_stream_text!r} != {raw_non_stream["content"]!r}'
    )
JsonlRecorder(
    DRIVE_RECORDS / 'days-03-04-raw-stream.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append(
    {
        'event': 'raw_stream_comparison',
        'request': {'prompt': stream_prompt, **raw_settings},
        'non_stream_response': raw_non_stream,
        'stream_events': raw_stream_events,
    }
)
print('Raw stream and non-stream text match')
print(raw_stream_text)

In [ ]:
from toolsmith.deltas import aggregate_chunks

sdk_stream = openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=stream_messages,
    tools=registry.to_openai(),
    tool_choice='auto',
    stream=True,
    stream_options={'include_usage': True},
    **generation,
)
sdk_chunks = [chunk.model_dump() for chunk in sdk_stream]
sdk_result = aggregate_chunks(sdk_chunks)
sdk_calls = sdk_result['message'].get('tool_calls', [])
if len(sdk_calls) != 1 or sdk_calls[0]['function']['name'] != 'get_weather':
    raise AssertionError(f'Unexpected streamed tool calls: {sdk_result}')
sdk_arguments = json.loads(sdk_calls[0]['function']['arguments'])
if sdk_arguments.get('city') != 'Paris':
    raise AssertionError(f'Unexpected streamed arguments: {sdk_arguments}')
if sdk_result['finish_reason'] != 'tool_calls':
    raise AssertionError(f'Unexpected finish reason: {sdk_result["finish_reason"]}')
JsonlRecorder(
    DRIVE_RECORDS / 'days-03-04-openai-stream.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append({'event': 'openai_sdk_stream', 'chunks': sdk_chunks, 'result': sdk_result})
print('Streamed tool call:', sdk_calls[0])
print('Usage:', sdk_result['usage'])
print('OpenAI SDK streaming check passed')

## Days 4–5: compile JSON Schemas into GBNF

Our compiler emits JSON syntax and Qwen3 envelopes directly. It supports closed objects, required/optional properties, scalar types, scalar enums, homogeneous arrays, and inline nesting. Every other validation keyword is rejected with a JSON Pointer instead of being silently ignored. Object properties use a deterministic order: required properties first, then optional properties in declaration order.

Grammar validity is only syntactic. We still validate decoded arguments against the selected tool's JSON Schema, and we separately record whether the model chose the semantically requested tool. In `auto`, plain text uses a marker-aware automaton so malformed `<tool_call>` output cannot slip through the text alternative. These constrained requests disable thinking; trigger-activated think-then-constrain decoding is implemented in Days 6–8 below.

The cells below first compile and save the corpus, then use llama.cpp's own `test-gbnf-validator` and converter, then exercise the OpenAI SDK and run a fixed-prompt ablation. The pinned validator returns exit code zero even for invalid inputs, so its explicit output verdict is checked.


In [ ]:
import json

from jsonschema import Draft202012Validator

from toolsmith.recording import JsonlRecorder
from toolsmith.grammar import GBNFCompiler, SchemaNotSupportedError, ToolChoice
from toolsmith.registry import Tool
from toolsmith.validation import ToolCallValidator

grammar_compiler = GBNFCompiler()
grammar_corpus = json.loads(
    (PROJECT_DIR / 'tests' / 'fixtures' / 'grammar_cases.json').read_text()
)
DRIVE_GRAMMARS = DRIVE_ROOT / 'grammars' / 'days-04-05'
DRIVE_GRAMMARS.mkdir(parents=True, exist_ok=True)
grammar_artifacts = {}
unsupported_results = []

for case in grammar_corpus['supported']:
    compiled = grammar_compiler.compile_schema(case['schema'])
    repeated = grammar_compiler.compile_schema(case['schema'])
    if compiled.text != repeated.text or compiled.sha256 != repeated.sha256:
        raise AssertionError(f"Non-deterministic grammar for {case['name']}")
    grammar_path = DRIVE_GRAMMARS / f"{case['name']}-toolsmith.gbnf"
    grammar_path.write_text(compiled.text, encoding='utf-8')
    validator = Draft202012Validator(case['schema'])
    for sample in case['valid']:
        validator.validate(json.loads(sample['text']))
    cases = [
        {**sample, 'expected': True}
        for sample in case['valid']
    ] + [
        {**sample, 'expected': False}
        for sample in case['invalid']
    ]
    grammar_artifacts[f"schema:{case['name']}"] = {
        'kind': 'schema',
        'name': case['name'],
        'schema': case['schema'],
        'grammar': compiled.text,
        'grammar_path': grammar_path,
        'grammar_sha256': compiled.sha256,
        'cases': cases,
    }

for case in grammar_corpus['unsupported']:
    try:
        grammar_compiler.compile_schema(case['schema'])
    except SchemaNotSupportedError as error:
        if error.keyword != case['keyword'] or error.pointer != case['pointer']:
            raise AssertionError(
                f"Wrong schema error for {case['name']}: "
                f"{error.keyword} at {error.pointer}"
            ) from error
        unsupported_results.append(
            {
                'name': case['name'],
                'keyword': error.keyword,
                'pointer': error.pointer,
                'message': str(error),
            }
        )
    else:
        raise AssertionError(f"Unsupported schema compiled: {case['name']}")

grammar_tools = [
    Tool('get_weather', 'Demonstration weather', grammar_corpus['supported'][0]['schema']),
    Tool(
        'search_docs',
        'Search documentation',
        {
            'type': 'object',
            'properties': {'query': {'type': 'string'}},
            'required': ['query'],
            'additionalProperties': False,
        },
    ),
    Tool('create_profile', 'Create a demonstration profile', grammar_corpus['supported'][1]['schema']),
]
grammar_tool_definitions = [tool.to_openai() for tool in grammar_tools]

def qwen_call(name, arguments):
    envelope = json.dumps(
        {'name': name, 'arguments': arguments},
        ensure_ascii=False,
        separators=(',', ':'),
    )
    return f'<tool_call>\n{envelope}\n</tool_call>'

weather_call = qwen_call('get_weather', {'city': 'Paris', 'unit': 'c'})
search_call = qwen_call('search_docs', {'query': 'constrained decoding'})
profile_call = qwen_call(
    'create_profile',
    {
        'name': 'Ada',
        'location': {'lat': 1.5, 'lon': -2},
        'tags': ['math', 'code'],
        'active': True,
    },
)
parallel_calls = weather_call + '\n' + search_call
marker_argument_call = qwen_call('get_weather', {'city': '</tool_call>'})
unknown_call = qwen_call('unknown', {})
wrong_arguments_call = qwen_call('get_weather', {'city': 3})

def add_tool_artifact(label, choice, valid, invalid):
    compiled = grammar_compiler.compile_tools(grammar_tools, choice)
    grammar_path = DRIVE_GRAMMARS / f'{label}-toolsmith.gbnf'
    grammar_path.write_text(compiled.text, encoding='utf-8')
    grammar_artifacts[f'tools:{label}'] = {
        'kind': 'tools',
        'name': label,
        'choice': choice,
        'grammar': compiled.text,
        'grammar_path': grammar_path,
        'grammar_sha256': compiled.sha256,
        'cases': [
            {'name': f'valid-{index}', 'text': text, 'expected': True}
            for index, text in enumerate(valid)
        ] + [
            {'name': f'invalid-{index}', 'text': text, 'expected': False}
            for index, text in enumerate(invalid)
        ],
    }

add_tool_artifact(
    'auto',
    'auto',
    [
        'plain café text',
        '2 < 3 and 4 > 1',
        '<<tool_call',
        weather_call,
        parallel_calls,
        marker_argument_call,
    ],
    [
        '<tool_call>{"name":"get_weather"',
        unknown_call,
        wrong_arguments_call,
        weather_call + ' trailing prose',
    ],
)
add_tool_artifact(
    'none',
    'none',
    ['No tool needed', '2 < 3', '<tool_'],
    [weather_call, '</tool_call>'],
)
add_tool_artifact(
    'required',
    'required',
    [weather_call, parallel_calls, profile_call, marker_argument_call],
    ['plain text', unknown_call, wrong_arguments_call, weather_call + ' trailing'],
)
named_choice = {'type': 'function', 'function': {'name': 'get_weather'}}
add_tool_artifact(
    'named-get-weather',
    named_choice,
    [weather_call, marker_argument_call],
    [parallel_calls, search_call, 'plain text'],
)

offline_record = {
    'event': 'grammar_offline_corpus',
    'artifacts': [
        {
            'key': key,
            'path': str(artifact['grammar_path']),
            'sha256': artifact['grammar_sha256'],
            'case_count': len(artifact['cases']),
        }
        for key, artifact in grammar_artifacts.items()
    ],
    'unsupported': unsupported_results,
}
JsonlRecorder(
    DRIVE_RECORDS / 'days-04-05-offline-corpus.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append(offline_record)
print(
    f"Compiled {len(grammar_artifacts)} grammars; "
    f"rejected {len(unsupported_results)} unsupported schemas; "
    f"artifacts: {DRIVE_GRAMMARS}"
)


In [ ]:
import tempfile

if not (LOCAL_SOURCE / '.git').is_dir():
    subprocess.run(
        [
            'git', 'clone', '--filter=blob:none',
            'https://github.com/ggml-org/llama.cpp.git', str(LOCAL_SOURCE),
        ],
        check=True,
    )
subprocess.run(
    ['git', '-C', str(LOCAL_SOURCE), 'fetch', '--depth', '1', 'origin', 'tag', LLAMA_CPP_REF],
    check=True,
)
subprocess.run(
    ['git', '-C', str(LOCAL_SOURCE), 'checkout', '--detach', LLAMA_CPP_REF],
    check=True,
)
GRAMMAR_BUILD = Path('/content/llama.cpp-gbnf-build')
subprocess.run(
    [
        'cmake', '-S', str(LOCAL_SOURCE), '-B', str(GRAMMAR_BUILD),
        '-DGGML_CUDA=OFF', '-DLLAMA_BUILD_TESTS=ON', '-DLLAMA_CURL=OFF',
        '-DCMAKE_BUILD_TYPE=Release',
    ],
    check=True,
)
subprocess.run(
    ['cmake', '--build', str(GRAMMAR_BUILD), '--target', 'test-gbnf-validator', '-j', '8'],
    check=True,
)
GBNF_VALIDATOR = GRAMMAR_BUILD / 'bin' / 'test-gbnf-validator'
if not GBNF_VALIDATOR.is_file():
    raise FileNotFoundError(GBNF_VALIDATOR)

validator_environment = os.environ.copy()
validator_paths = [str(GRAMMAR_BUILD / 'lib'), str(GRAMMAR_BUILD / 'bin')]
if validator_environment.get('LD_LIBRARY_PATH'):
    validator_paths.append(validator_environment['LD_LIBRARY_PATH'])
validator_environment['LD_LIBRARY_PATH'] = os.pathsep.join(validator_paths)

def validate_gbnf(grammar_path, input_text):
    with tempfile.TemporaryDirectory(prefix='toolsmith-gbnf-') as temporary:
        input_path = Path(temporary) / 'input.txt'
        input_path.write_text(input_text, encoding='utf-8')
        result = subprocess.run(
            [str(GBNF_VALIDATOR), str(grammar_path), str(input_path)],
            cwd=str(GBNF_VALIDATOR.parent),
            env=validator_environment,
            capture_output=True,
            text=True,
            timeout=30,
        )
    output = result.stdout + result.stderr
    if result.returncode != 0:
        raise RuntimeError(f'GBNF validator failed ({result.returncode}): {output}')
    if 'Input string is valid according to the grammar.' in output:
        return True, output.strip()
    if 'Input string is invalid according to the grammar.' in output:
        return False, output.strip()
    raise RuntimeError(f'No explicit GBNF verdict: {output}')

converter_path = LOCAL_SOURCE / 'examples' / 'json_schema_to_grammar.py'
validation_recorder = JsonlRecorder(
    DRIVE_RECORDS / 'days-04-05-validator-comparison.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
grammar_validation_summary = []
reference_differences = []

for key, artifact in grammar_artifacts.items():
    own_results = []
    for sample in artifact['cases']:
        accepted, verdict = validate_gbnf(artifact['grammar_path'], sample['text'])
        own_results.append(
            {
                'name': sample['name'],
                'expected': sample['expected'],
                'accepted': accepted,
                'verdict': verdict,
            }
        )
        if accepted != sample['expected']:
            raise AssertionError(
                f"Our GBNF mismatch for {key}/{sample['name']}: "
                f"expected {sample['expected']}, got {accepted}\n{verdict}"
            )

    reference_result = None
    if artifact['kind'] == 'schema':
        with tempfile.TemporaryDirectory(prefix='toolsmith-schema-') as temporary:
            schema_path = Path(temporary) / 'schema.json'
            schema_path.write_text(json.dumps(artifact['schema']), encoding='utf-8')
            converted = subprocess.run(
                [sys.executable, str(converter_path), str(schema_path)],
                capture_output=True,
                text=True,
                timeout=30,
            )
        if converted.returncode != 0:
            raise RuntimeError(
                f"Reference converter failed for {key}: {converted.stdout}{converted.stderr}"
            )
        reference_path = DRIVE_GRAMMARS / f"{artifact['name']}-llama-cpp.gbnf"
        reference_path.write_text(converted.stdout, encoding='utf-8')
        reference_cases = []
        for sample in artifact['cases']:
            accepted, verdict = validate_gbnf(reference_path, sample['text'])
            reference_cases.append(
                {
                    'name': sample['name'],
                    'expected': sample['expected'],
                    'accepted': accepted,
                    'verdict': verdict,
                }
            )
            if accepted != sample['expected']:
                difference = {
                    'artifact': key,
                    'sample': sample['name'],
                    'expected': sample['expected'],
                    'reference_accepted': accepted,
                    'verdict': verdict,
                }
                reference_differences.append(difference)
                if sample.get('differential', True):
                    raise AssertionError(f'Reference differential mismatch: {difference}')
                print(f"Documented reference difference: {key}/{sample['name']}")
        reference_result = {
            'path': str(reference_path),
            'warnings': converted.stderr.strip(),
            'cases': reference_cases,
        }

    record = {
        'event': 'gbnf_validation',
        'artifact': key,
        'grammar_sha256': artifact['grammar_sha256'],
        'grammar_path': str(artifact['grammar_path']),
        'own_results': own_results,
        'reference': reference_result,
    }
    validation_recorder.append(record)
    grammar_validation_summary.append(record)
    print(f"{key}: {len(own_results)} own cases passed")

print(
    f"llama.cpp validation passed for {len(grammar_artifacts)} grammars; "
    f"reference differences: {len(reference_differences)}"
)


In [ ]:
from toolsmith.api import ChatCompletionRequest, ToolsmithService
from toolsmith.deltas import aggregate_chunks

sdk_grammar_recorder = JsonlRecorder(
    DRIVE_RECORDS / 'days-04-05-tool-choice.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
grammar_validators = {
    tool.name: Draft202012Validator(tool.parameters) for tool in grammar_tools
}

def validate_sdk_calls(calls):
    decoded = []
    for call in calls:
        name = call.function.name
        arguments = json.loads(call.function.arguments)
        grammar_validators[name].validate(arguments)
        decoded.append({'name': name, 'arguments': arguments})
    return decoded

def validate_aggregated_calls(calls):
    decoded = []
    for call in calls:
        name = call['function']['name']
        arguments = json.loads(call['function']['arguments'])
        grammar_validators[name].validate(arguments)
        decoded.append({'name': name, 'arguments': arguments})
    return decoded

def sdk_grammar_request(label, prompt_text, choice):
    response = openai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{'role': 'user', 'content': prompt_text}],
        tools=grammar_tool_definitions,
        tool_choice=choice,
        max_tokens=256,
        temperature=0.0,
        seed=0,
        extra_body={'chat_template_kwargs': {'enable_thinking': False}},
    )
    message = response.choices[0].message
    calls = validate_sdk_calls(message.tool_calls or [])
    record = {
        'event': 'sdk_tool_choice',
        'label': label,
        'choice': choice,
        'response': response.model_dump(),
        'decoded_calls': calls,
    }
    sdk_grammar_recorder.append(record)
    print(label, response.choices[0].finish_reason, calls or message.content)
    return response, calls

auto_text_response, auto_text_calls = sdk_grammar_request(
    'auto-text',
    'What is 2 + 2? Answer directly without using a tool.',
    'auto',
)
if auto_text_calls or not auto_text_response.choices[0].message.content:
    raise AssertionError('auto irrelevance check should return plain text')

auto_call_response, auto_calls = sdk_grammar_request(
    'auto-call',
    'Use get_weather exactly once for Paris in Celsius.',
    'auto',
)
if [call['name'] for call in auto_calls] != ['get_weather']:
    raise AssertionError(f'Unexpected auto calls: {auto_calls}')

none_response, none_calls = sdk_grammar_request(
    'none',
    'Use get_weather for Paris, but tool_choice forbids calls; explain what you can do.',
    'none',
)
if none_calls or not none_response.choices[0].message.content:
    raise AssertionError('tool_choice=none returned a call or empty text')

required_response, required_calls = sdk_grammar_request(
    'required',
    'Call get_weather for Paris in Celsius.',
    'required',
)
if not required_calls:
    raise AssertionError('tool_choice=required returned no calls')

named_response, named_calls = sdk_grammar_request(
    'named',
    'Find documentation about constrained decoding.',
    {'type': 'function', 'function': {'name': 'search_docs'}},
)
if len(named_calls) != 1 or named_calls[0]['name'] != 'search_docs':
    raise AssertionError(f'Named choice mismatch: {named_calls}')

def sdk_grammar_stream(label, prompt_text, choice, expected_name):
    stream = openai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{'role': 'user', 'content': prompt_text}],
        tools=grammar_tool_definitions,
        tool_choice=choice,
        stream=True,
        stream_options={'include_usage': True},
        max_tokens=256,
        temperature=0.0,
        seed=0,
        extra_body={'chat_template_kwargs': {'enable_thinking': False}},
    )
    chunks = [chunk.model_dump() for chunk in stream]
    result = aggregate_chunks(chunks)
    calls = validate_aggregated_calls(result['message'].get('tool_calls', []))
    if len(calls) != 1 or calls[0]['name'] != expected_name:
        raise AssertionError(f'{label} call mismatch: {result}')
    if result['finish_reason'] != 'tool_calls' or result.get('usage') is None:
        raise AssertionError(f'{label} finish/usage mismatch: {result}')
    sdk_grammar_recorder.append(
        {
            'event': 'sdk_tool_choice_stream',
            'label': label,
            'choice': choice,
            'chunks': chunks,
            'result': result,
            'decoded_calls': calls,
        }
    )
    print(label, calls, result['usage'])

sdk_grammar_stream(
    'required-stream',
    'Call get_weather for Paris in Celsius.',
    'required',
    'get_weather',
)
sdk_grammar_stream(
    'named-stream',
    'Find documentation about constrained decoding.',
    {'type': 'function', 'function': {'name': 'search_docs'}},
    'search_docs',
)

class MalformedStreamClient:
    def stream_complete(self, prompt, **settings):
        yield {'content': '<tool_call>', 'stop': False}
        yield {'content': '', 'stop': True, 'truncated': False}

malformed_service = ToolsmithService(MalformedStreamClient(), renderer)
malformed_request = ChatCompletionRequest(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': 'Call get_weather.'}],
    tools=grammar_tool_definitions,
    tool_choice='required',
    stream=True,
    chat_template_kwargs={'enable_thinking': False},
)
malformed_parts = list(malformed_service.chat_stream(malformed_request))
malformed_payloads = [
    json.loads(part[len('data: '):])
    for part in malformed_parts
    if part.startswith('data: ') and part != 'data: [DONE]\n\n'
]
if not any(
    payload.get('error', {}).get('code') == 'unclosed_tool_call'
    for payload in malformed_payloads
):
    raise AssertionError(f'Malformed stream did not fail explicitly: {malformed_payloads}')
if any(
    choice.get('finish_reason') is not None
    for payload in malformed_payloads
    for choice in payload.get('choices', [])
):
    raise AssertionError('Malformed stream emitted a false success finish')
sdk_grammar_recorder.append(
    {'event': 'malformed_stream', 'payloads': malformed_payloads}
)
print('OpenAI SDK tool-choice and malformed-stream checks passed')


In [ ]:
import statistics

from toolsmith.backends import CompletionTruncatedError
from toolsmith.parsing import Qwen3ToolCallParser, ToolCallParseError
from toolsmith.validation import ToolCallValidationError

ablation_prompts = [
    {
        'name': 'simple',
        'prompt': 'Use get_weather exactly once for Paris in Celsius.',
        'expected': ['get_weather'],
    },
    {
        'name': 'nested-array-enum',
        'prompt': (
            'Use create_profile exactly once with name Ada, tags math and code, '
            'location lat 1.5 and lon -2, active true, count -3, and nothing null.'
        ),
        'expected': ['create_profile'],
    },
    {
        'name': 'parallel',
        'prompt': (
            'Make two calls: get_weather for Paris in Celsius and '
            'search_docs for constrained decoding.'
        ),
        'expected': ['get_weather', 'search_docs'],
    },
    {
        'name': 'no-tool-needed',
        'prompt': 'What is 2 + 2? Answer directly without using a tool.',
        'expected': [],
    },
]
compile_times = []
for _ in range(5):
    compile_started = time.perf_counter()
    ablation_compiled = grammar_compiler.compile_tools(grammar_tools, 'auto')
    compile_times.append(time.perf_counter() - compile_started)
ablation_grammar = ablation_compiled.text
ablation_validator = ToolCallValidator(grammar_tools, ToolChoice('auto'))
ablation_recorder = JsonlRecorder(
    DRIVE_RECORDS / 'days-04-05-ablation.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
ablation_results = []
constrained_failures = []

for case in ablation_prompts:
    messages = [{'role': 'user', 'content': case['prompt']}]
    rendered = renderer.render(
        messages,
        grammar_tool_definitions,
        enable_thinking=False,
    )
    for mode in ('prompt-only', 'own-gbnf'):
        settings = {
            'max_tokens': 384,
            'temperature': 0.0,
            'seed': 0,
        }
        if mode == 'own-gbnf':
            settings['grammar'] = ablation_grammar
        started = time.perf_counter()
        outcome = 'ok'
        try:
            raw = client.complete(rendered, **settings)
        except CompletionTruncatedError as error:
            raw = error.response
            outcome = 'truncated'
        elapsed = time.perf_counter() - started
        content = raw.get('content', '')
        parsed = None
        parse_error = None
        validation_error = None
        calls = []
        schema_valid = False
        if outcome == 'ok':
            try:
                parsed = Qwen3ToolCallParser().parse(content)
                ablation_validator.validate(parsed)
                schema_valid = True
                calls = [
                    {
                        'name': call.name,
                        'arguments': json.loads(call.arguments),
                    }
                    for call in parsed.tool_calls
                ]
                if not parsed.content and not parsed.tool_calls:
                    validation_error = 'empty_model_response'
            except ToolCallParseError as error:
                parse_error = f'{error.code}: {error}'
            except ToolCallValidationError as error:
                validation_error = f'{error.code}: {error}'
        names = [call['name'] for call in calls]
        if case['name'] == 'parallel':
            choice_correct = len(names) == 2 and set(names) == set(case['expected'])
        else:
            choice_correct = names == case['expected']
        timings = raw.get('timings') if isinstance(raw.get('timings'), dict) else {}
        completion_tokens = int(
            timings.get('predicted_n', raw.get('tokens_predicted', 0)) or 0
        )
        tokens_per_second = timings.get('predicted_per_second')
        if tokens_per_second is None and elapsed > 0:
            tokens_per_second = completion_tokens / elapsed
        record = {
            'event': 'grammar_ablation',
            'case': case['name'],
            'mode': mode,
            'request': {'prompt': rendered, **settings},
            'raw_response': raw,
            'outcome': outcome,
            'elapsed_seconds': elapsed,
            'parse_error': parse_error,
            'validation_error': validation_error,
            'parse_rate': int(parsed is not None and parse_error is None),
            'schema_valid': schema_valid,
            'calls': calls,
            'choice_correct': choice_correct,
            'false_call': case['name'] == 'no-tool-needed' and bool(calls),
            'completion_tokens': completion_tokens,
            'tokens_per_second': tokens_per_second,
            'grammar_sha256': ablation_compiled.sha256 if mode == 'own-gbnf' else None,
            'grammar_bytes': len(ablation_grammar.encode('utf-8')) if mode == 'own-gbnf' else 0,
            'grammar_rules': ablation_compiled.rule_count if mode == 'own-gbnf' else 0,
            'grammar_compile_seconds_median': statistics.median(compile_times),
            'grammar_compile_seconds_max': max(compile_times),
        }
        ablation_recorder.append(record)
        ablation_results.append(record)
        if mode == 'own-gbnf' and (
            outcome != 'ok' or parse_error or validation_error or not schema_valid
        ):
            constrained_failures.append(record)
        print(
            f"{case['name']:18} {mode:11} outcome={outcome:9} "
            f"parse={record['parse_rate']} schema={int(schema_valid)} "
            f"semantic={int(choice_correct)} calls={names}"
        )

if constrained_failures:
    raise AssertionError(
        f"Constrained generations failed structural checks: {constrained_failures}"
    )
print(
    f"Ablation records: {DRIVE_RECORDS / 'days-04-05-ablation.jsonl'}; "
    f"median compile time: {statistics.median(compile_times) * 1000:.2f} ms"
)


## Days 6–8: vocabulary trie, automaton, and token masking

The GBNF compiler now builds an immutable grammar IR. The same IR renders llama.cpp's GBNF and compiles to our pushdown automaton, so there is no second GBNF parser. Persistent return stacks support nested JSON and nondeterministic branches; tail calls keep long plain text and strings from growing the stack.

Qwen tokens are treated as raw bytes. An incremental UTF-8 decoder feeds codepoints to the automaton, allowing a token to end inside a multi-byte character while preventing EOS there. The mask engine walks the vocabulary byte trie and automaton together, pruning impossible prefixes and testing every token as a whole. A brute-force per-token mask is the correctness oracle.

Fixed masks constrain the entire response. Triggered masks leave reasoning free until `<tool_call>`, but candidate tokens that complete the marker are checked atomically with any following bytes. Thus a marker-plus-invalid-arguments token cannot be emitted before constraints activate. This section uses Qwen3-0.6B through Transformers. The Qwen3-8B llama.cpp server is stopped first so both models are not resident on a 16 GB T4, and it is started again before the Day 9 live API calls.


In [ ]:
import random

from toolsmith.sampling.automaton import PushdownAutomaton
from toolsmith.sampling.mask import TokenMaskEngine
from toolsmith.sampling.vocabulary import Vocabulary, VocabularyEntry

sampler_offline_results = []
for key, artifact in grammar_artifacts.items():
    if artifact['kind'] == 'schema':
        compiled = grammar_compiler.compile_schema(artifact['schema'])
    else:
        compiled = grammar_compiler.compile_tools(grammar_tools, artifact['choice'])
    if compiled.sha256 != artifact['grammar_sha256']:
        raise AssertionError(f'Grammar hash changed for {key}')
    automaton = PushdownAutomaton.from_compiled(compiled)
    case_results = []
    for sample in artifact['cases']:
        data = sample['text'].encode('utf-8')
        state = automaton.initial_state()
        valid_prefix = True
        if sample['expected']:
            for byte in data:
                state = automaton.advance_bytes(state, bytes((byte,)))
                if state is None:
                    valid_prefix = False
                    break
            accepted = valid_prefix and automaton.can_finish(state)
            if not accepted:
                raise AssertionError(f"PDA rejected {key}/{sample['name']}")
        else:
            state = automaton.advance_bytes(state, data)
            accepted = state is not None and automaton.can_finish(state)
            if accepted:
                raise AssertionError(f"PDA accepted invalid {key}/{sample['name']}")
        if sample['expected'] and len(data) > 1:
            randomizer = random.Random(31)
            for _ in range(3):
                chunked = automaton.initial_state()
                offset = 0
                while offset < len(data):
                    size = randomizer.randint(1, 9)
                    chunked = automaton.advance_bytes(chunked, data[offset:offset + size])
                    if chunked is None:
                        raise AssertionError(f"Chunk mismatch for {key}/{sample['name']}")
                    offset += size
                if not automaton.can_finish(chunked):
                    raise AssertionError(f"Chunk finalization mismatch for {key}/{sample['name']}")
        case_results.append({'name': sample['name'], 'expected': sample['expected']})
    sampler_offline_results.append(
        {'artifact': key, 'sha256': compiled.sha256, 'cases': case_results}
    )
    print(f"{key}: {len(case_results)} PDA cases passed")

synthetic_call = weather_call.encode('utf-8')
synthetic_tokens = [
    b'<tool_call>', b'<tool_call>bad', b'not-json', b' ',
    *(bytes((byte,)) for byte in sorted(set(synthetic_call))),
]
synthetic_vocabulary = Vocabulary(
    [VocabularyEntry(index, raw) for index, raw in enumerate(synthetic_tokens)],
    model_vocabulary_size=len(synthetic_tokens) + 2,
    tokenizer_vocabulary_size=len(synthetic_tokens) + 1,
    eos_token_ids=frozenset({len(synthetic_tokens) + 1}),
    special_token_ids=frozenset({len(synthetic_tokens) + 1}),
    excluded_empty_tokens=0,
    build_seconds=0,
)
synthetic_compiled = grammar_compiler.compile_tools(grammar_tools[:1], 'required')
synthetic_automaton = PushdownAutomaton.from_compiled(synthetic_compiled)
synthetic_engine = TokenMaskEngine(synthetic_vocabulary, synthetic_automaton)
synthetic_prefixes = 0
for offset in range(len(synthetic_call) + 1):
    state = synthetic_automaton.advance_bytes(
        synthetic_automaton.initial_state(), synthetic_call[:offset]
    )
    if state is None:
        raise AssertionError(f'Synthetic prefix died at {offset}')
    optimized = synthetic_engine.mask(state).allowed
    brute = synthetic_engine.brute_force_mask(state)
    if optimized != brute:
        raise AssertionError(f'Mask mismatch at prefix {offset}: {optimized} != {brute}')
    synthetic_prefixes += 1

sampler_offline_record = {
    'event': 'sampler_offline_differential',
    'artifacts': sampler_offline_results,
    'artifact_count': len(sampler_offline_results),
    'case_count': sum(len(item['cases']) for item in sampler_offline_results),
    'synthetic_mask_prefixes': synthetic_prefixes,
}
JsonlRecorder(
    DRIVE_RECORDS / 'days-06-08-automaton.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append(sampler_offline_record)
print(
    f"PDA validated {sampler_offline_record['case_count']} cases; "
    f"trie/brute masks matched at {synthetic_prefixes} prefixes"
)


In [ ]:
import unicodedata

from transformers import AutoConfig, AutoTokenizer

from toolsmith.sampling.mask import TokenMaskEngine
from toolsmith.sampling.vocabulary import Vocabulary

SAMPLER_MODEL_NAME = 'Qwen/Qwen3-0.6B'
sampler_config = AutoConfig.from_pretrained(SAMPLER_MODEL_NAME, token=HF_TOKEN)
sampler_tokenizer = AutoTokenizer.from_pretrained(
    SAMPLER_MODEL_NAME,
    token=HF_TOKEN,
    use_fast=False,
)
sampler_eos = sampler_config.eos_token_id or sampler_tokenizer.eos_token_id
if isinstance(sampler_eos, (list, tuple, set)):
    sampler_eos_ids = frozenset(sampler_eos)
else:
    sampler_eos_ids = frozenset({sampler_eos})
sampler_vocabulary = Vocabulary.from_qwen_tokenizer(
    sampler_tokenizer,
    model_vocabulary_size=sampler_config.vocab_size,
    eos_token_ids=sampler_eos_ids,
)
vocabulary_audits = []
for text in (
    'plain café text',
    '2 < 3 and 4 > 1',
    '雪😀 Zürich',
    '{"city":"Zürich","unit":"c"}',
    '<tool_call>\n{"name":"get_weather"}\n</tool_call>',
):
    normalized = unicodedata.normalize('NFC', text)
    ids = sampler_tokenizer(normalized, add_special_tokens=False)['input_ids']
    reconstructed = sampler_vocabulary.reconstruct_bytes(ids)
    if reconstructed != normalized.encode('utf-8'):
        raise AssertionError(f'Raw-byte vocabulary round trip failed for {text!r}')
    vocabulary_audits.append(
        {'text': text, 'token_ids': ids, 'raw_hex': reconstructed.hex()}
    )
for marker in ('<tool_call>', '</tool_call>', '<think>', '</think>'):
    token_id = sampler_tokenizer.convert_tokens_to_ids(marker)
    if sampler_vocabulary.raw_bytes(token_id) != marker.encode('utf-8'):
        raise AssertionError(f'Added-token byte mismatch: {marker}')

real_compiled = grammar_compiler.compile_tools(grammar_tools[:1], 'required')
real_automaton = PushdownAutomaton.from_compiled(real_compiled)
real_engine = TokenMaskEngine(sampler_vocabulary, real_automaton)
real_mask_checks = []
weather_bytes = weather_call.encode('utf-8')
positions = sorted(set([0, 1, 5, len(b'<tool_call>'), len(weather_bytes) // 3, len(weather_bytes) // 2, len(weather_bytes) - 1]))
for position in positions:
    state = real_automaton.advance_bytes(
        real_automaton.initial_state(), weather_bytes[:position]
    )
    optimized = real_engine.mask(state).allowed
    brute = real_engine.brute_force_mask(state)
    if optimized != brute:
        raise AssertionError(f'Real vocabulary mask mismatch at byte {position}')
    real_mask_checks.append(
        {'byte_offset': position, 'allowed_tokens': len(optimized)}
    )
    print(f'Real-vocabulary mask oracle passed at byte {position}')

sampler_vocabulary_record = {
    'event': 'qwen_vocabulary',
    'model': SAMPLER_MODEL_NAME,
    'revision': getattr(sampler_config, '_commit_hash', None),
    'fingerprint': sampler_vocabulary.fingerprint,
    'stats': sampler_vocabulary.stats.as_dict(),
    'audits': vocabulary_audits,
    'real_mask_checks': real_mask_checks,
}
JsonlRecorder(
    DRIVE_RECORDS / 'days-06-08-vocabulary.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append(sampler_vocabulary_record)
print(json.dumps(sampler_vocabulary_record['stats'], indent=2))


In [ ]:
import torch
from transformers import AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError('The Qwen sampler acceptance run requires a CUDA runtime')
running_server = globals().get('server')
if running_server is not None and running_server.is_running:
    print('Stopping Qwen3-8B llama-server so Qwen3-0.6B can use the GPU.', flush=True)
    running_server.stop(timeout=20)
if 'torch' in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f'CUDA free {free_bytes / 2**30:.2f} GiB / total {total_bytes / 2**30:.2f} GiB')
if free_bytes < 2 * 1024**3:
    raise RuntimeError('Less than 2 GiB CUDA memory is free for Qwen3-0.6B; inspect prior cells')
sampler_model = AutoModelForCausalLM.from_pretrained(
    SAMPLER_MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
).to('cuda')
sampler_model.eval()
if sampler_model.config.vocab_size != sampler_vocabulary.model_vocabulary_size:
    raise AssertionError('Model/tokenizer padded vocabulary sizes differ')
print(
    f'Loaded {SAMPLER_MODEL_NAME}; device={sampler_model.device}; '
    f'dtype={next(sampler_model.parameters()).dtype}'
)


In [ ]:
from dataclasses import asdict

from toolsmith.grammar import resolve_tool_choice
from toolsmith.parsing import Qwen3ToolCallParser, ToolCallParseError
from toolsmith.sampling.controller import (
    FixedMaskController,
    IncompleteConstrainedGeneration,
)
from toolsmith.sampling.hf import (
    ConstrainedLogitsProcessor,
    generate_constrained,
    generate_unconstrained,
)
from toolsmith.validation import ToolCallValidationError

fixed_recorder = JsonlRecorder(
    DRIVE_RECORDS / 'days-06-08-fixed-sampler.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
fixed_cases = [
    ('simple', 'Use get_weather exactly once for Paris in Celsius.', 'required'),
    (
        'nested-array-enum',
        'Use create_profile exactly once with name Ada, tags math and code, '
        'location lat 1.5 and lon -2, active true, count -3, and nothing null.',
        'required',
    ),
    (
        'parallel',
        'Make two calls: get_weather for Paris in Celsius and search_docs for constrained decoding.',
        'required',
    ),
    (
        'named',
        'Find documentation about constrained decoding.',
        {'type': 'function', 'function': {'name': 'search_docs'}},
    ),
]
fixed_results = []
constrained_failures = []
benchmark_mask_seconds = []
benchmark_controllers = []

def result_metrics(result):
    return {
        'token_ids': list(result.token_ids),
        'stop_reason': result.stop_reason,
        'prompt_tokens': result.prompt_tokens,
        'elapsed_seconds': result.elapsed_seconds,
        'tokens_per_second': result.tokens_per_second,
        'mask_calls': len(result.mask_seconds),
        'cache_info': result.cache_info,
    }

for name, prompt_text, selected_choice in fixed_cases:
    rendered = renderer.__class__(sampler_tokenizer).render(
        [{'role': 'user', 'content': prompt_text}],
        grammar_tool_definitions,
        enable_thinking=False,
    )
    for mode in ('unconstrained', 'fixed'):
        result = None
        try:
            if mode == 'unconstrained':
                result = generate_unconstrained(
                    sampler_model,
                    sampler_tokenizer,
                    sampler_vocabulary,
                    rendered,
                    max_new_tokens=256,
                    seed=0,
                )
                parsed = None
                validation_error = None
                try:
                    parsed = Qwen3ToolCallParser(expect_thinking=False).parse(result.text)
                    ToolCallValidator(
                        grammar_tools, resolve_tool_choice(grammar_tools, selected_choice)
                    ).validate(parsed)
                except (ToolCallParseError, ToolCallValidationError) as error:
                    validation_error = str(error)
            else:
                compiled = grammar_compiler.compile_tools(grammar_tools, selected_choice)
                automaton = PushdownAutomaton.from_compiled(compiled)
                controller = FixedMaskController(sampler_vocabulary, automaton)
                result = generate_constrained(
                    sampler_model,
                    sampler_tokenizer,
                    sampler_vocabulary,
                    rendered,
                    controller,
                    max_new_tokens=256,
                    seed=0,
                )
                parsed = Qwen3ToolCallParser(expect_thinking=False).parse(result.text)
                ToolCallValidator(
                    grammar_tools, resolve_tool_choice(grammar_tools, selected_choice)
                ).validate(parsed)
                validation_error = None
                benchmark_mask_seconds.extend(result.mask_seconds)
                benchmark_controllers.append(controller)
            record = {
                'event': 'fixed_sampler',
                'case': name,
                'mode': mode,
                'choice': selected_choice,
                'raw_text': result.text,
                'raw_hex': result.raw_bytes.hex(),
                'parsed': asdict(parsed) if parsed is not None else None,
                'semantic_tool_names': (
                    [call.name for call in parsed.tool_calls]
                    if parsed is not None else []
                ),
                'validation_error': validation_error,
                **result_metrics(result),
            }
            if mode == 'fixed' and validation_error:
                constrained_failures.append(record)
        except (IncompleteConstrainedGeneration, ToolCallParseError, ToolCallValidationError, UnicodeDecodeError) as error:
            record = {
                'event': 'fixed_sampler_error',
                'case': name,
                'mode': mode,
                'choice': selected_choice,
                'error_type': type(error).__name__,
                'error': str(error),
                'raw_hex': (
                    result.raw_bytes.hex() if result is not None
                    else getattr(error, 'raw_bytes', b'').hex()
                ),
            }
            if mode == 'fixed':
                constrained_failures.append(record)
        fixed_recorder.append(record)
        fixed_results.append(record)
        print(name, mode, record.get('stop_reason', record['event']), record.get('validation_error'))

simple_prompt = renderer.__class__(sampler_tokenizer).render(
    [{'role': 'user', 'content': fixed_cases[0][1]}],
    grammar_tool_definitions,
    enable_thinking=False,
)
simple_compiled = grammar_compiler.compile_tools(grammar_tools, 'required')
cached_controller = FixedMaskController(
    sampler_vocabulary,
    PushdownAutomaton.from_compiled(simple_compiled),
    cache_size=4096,
)
uncached_controller = FixedMaskController(
    sampler_vocabulary,
    PushdownAutomaton.from_compiled(simple_compiled),
    cache_size=0,
)
cached_result = generate_constrained(
    sampler_model, sampler_tokenizer, sampler_vocabulary, simple_prompt,
    cached_controller, max_new_tokens=256, seed=0,
)
uncached_result = generate_constrained(
    sampler_model, sampler_tokenizer, sampler_vocabulary, simple_prompt,
    uncached_controller, max_new_tokens=256, seed=0,
)
if cached_result.token_ids != uncached_result.token_ids:
    raise AssertionError('Cache-on/cache-off greedy token IDs differ')
benchmark_mask_seconds.extend(cached_result.mask_seconds + uncached_result.mask_seconds)
benchmark_controllers.extend([cached_controller, uncached_controller])
fixed_recorder.append(
    {
        'event': 'cache_equivalence',
        'token_ids': list(cached_result.token_ids),
        'cached': result_metrics(cached_result),
        'uncached': result_metrics(uncached_result),
    }
)

sampled_controller = FixedMaskController(
    sampler_vocabulary,
    PushdownAutomaton.from_compiled(simple_compiled),
)
sampled_result = generate_constrained(
    sampler_model, sampler_tokenizer, sampler_vocabulary, simple_prompt,
    sampled_controller, max_new_tokens=256, do_sample=True,
    temperature=0.7, seed=17,
)
sampled_parsed = Qwen3ToolCallParser(expect_thinking=False).parse(sampled_result.text)
ToolCallValidator(grammar_tools, ToolChoice('required')).validate(sampled_parsed)
fixed_recorder.append(
    {
        'event': 'seeded_sampling',
        'raw_text': sampled_result.text,
        'raw_hex': sampled_result.raw_bytes.hex(),
        'parsed': asdict(sampled_parsed),
        **result_metrics(sampled_result),
    }
)
benchmark_mask_seconds.extend(sampled_result.mask_seconds)
benchmark_controllers.append(sampled_controller)

short_controller = FixedMaskController(
    sampler_vocabulary,
    PushdownAutomaton.from_compiled(simple_compiled),
)
try:
    generate_constrained(
        sampler_model, sampler_tokenizer, sampler_vocabulary, simple_prompt,
        short_controller, max_new_tokens=2, seed=0,
    )
except IncompleteConstrainedGeneration as error:
    fixed_recorder.append(
        {
            'event': 'explicit_truncation',
            'token_ids': list(error.token_ids),
            'raw_hex': error.raw_bytes.hex(),
            'state': error.state,
        }
    )
else:
    raise AssertionError('Two-token required generation did not report truncation')

invalid_logit_controller = FixedMaskController(
    sampler_vocabulary,
    PushdownAutomaton.from_compiled(simple_compiled),
)
invalid_processor = ConstrainedLogitsProcessor(invalid_logit_controller)
probe_ids = torch.tensor([[0, 1]], device=sampler_model.device)
probe_scores = torch.full(
    (1, sampler_model.config.vocab_size),
    -10.0,
    device=sampler_model.device,
    dtype=next(sampler_model.parameters()).dtype,
)
invalid_eos = next(iter(sampler_vocabulary.eos_token_ids))
probe_scores[0, invalid_eos] = 1000.0
masked_scores = invalid_processor(probe_ids, probe_scores)
if not torch.isneginf(masked_scores[0, invalid_eos]).item():
    raise AssertionError('Invalid highest-logit EOS was not masked')
if masked_scores[0, invalid_eos].item() == masked_scores[0].max().item():
    raise AssertionError('Invalid highest-logit token remains selected')

if constrained_failures:
    raise AssertionError(f'Fixed constrained generation failures: {constrained_failures}')
print('Fixed generation, cache equivalence, and explicit truncation checks passed')


In [ ]:
from toolsmith.sampling.controller import TriggeredMaskController

trigger_recorder = JsonlRecorder(
    DRIVE_RECORDS / 'days-06-08-trigger-sampler.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
trigger_automaton = PushdownAutomaton.from_compiled(
    grammar_compiler.compile_tools(grammar_tools, 'required')
)
trigger_cases = [
    (
        'thinking-call',
        'Think briefly, then call get_weather exactly once for Paris in Celsius.',
        True,
    ),
    (
        'thinking-no-tool',
        'Think briefly, then answer directly: what is 2 + 2? Do not call a tool.',
        False,
    ),
]
trigger_results = []
for name, prompt_text, expect_call in trigger_cases:
    prompt = renderer.__class__(sampler_tokenizer).render(
        [{'role': 'user', 'content': prompt_text}],
        grammar_tool_definitions,
        enable_thinking=True,
    )
    controller = TriggeredMaskController(sampler_vocabulary, trigger_automaton)
    result = generate_constrained(
        sampler_model,
        sampler_tokenizer,
        sampler_vocabulary,
        prompt,
        controller,
        max_new_tokens=512,
        seed=0,
    )
    parsed = Qwen3ToolCallParser(expect_thinking=None).parse(result.text)
    ToolCallValidator(grammar_tools, ToolChoice('auto')).validate(parsed)
    if expect_call and (controller.activation_step is None or not parsed.tool_calls):
        raise AssertionError(f'Thinking tool call did not trigger: {result.text!r}')
    if not expect_call and parsed.tool_calls:
        raise AssertionError(f'Irrelevance prompt produced a call: {parsed.tool_calls}')
    marker_position = result.raw_bytes.find(b'<tool_call>')
    prefix = (
        result.raw_bytes[:marker_position].decode('utf-8')
        if marker_position >= 0
        else result.text
    )
    record = {
        'event': 'triggered_sampler',
        'case': name,
        'raw_text': result.text,
        'raw_hex': result.raw_bytes.hex(),
        'free_prefix': prefix,
        'activation_step': controller.activation_step,
        'parsed': asdict(parsed),
        **result_metrics(result),
    }
    trigger_recorder.append(record)
    trigger_results.append(record)
    benchmark_mask_seconds.extend(result.mask_seconds)
    benchmark_controllers.append(controller)
    print(name, 'activation', controller.activation_step, 'calls', len(parsed.tool_calls))
print('Triggered thinking and no-tool generation checks passed')


In [ ]:
import math
import statistics

benchmark_engine = TokenMaskEngine(
    sampler_vocabulary,
    real_automaton,
    cache_size=4096,
)
benchmark_states = []
for position in range(0, len(weather_bytes) + 1, max(1, len(weather_bytes) // 24)):
    state = real_automaton.advance_bytes(
        real_automaton.initial_state(), weather_bytes[:position]
    )
    if state is not None:
        benchmark_states.append(state)
if not benchmark_states:
    raise AssertionError('No benchmark automaton states')
for index in range(max(200, 200 - len(benchmark_mask_seconds))):
    state = benchmark_states[index % len(benchmark_states)]
    started = time.perf_counter()
    benchmark_engine.mask(state)
    benchmark_mask_seconds.append(time.perf_counter() - started)

def percentile(values, probability):
    ordered = sorted(values)
    return ordered[max(0, math.ceil(probability * len(ordered)) - 1)]

constrained_runs = [
    record for record in fixed_results
    if record.get('event') == 'fixed_sampler' and record.get('mode') == 'fixed'
]
unconstrained_runs = [
    record for record in fixed_results
    if record.get('event') == 'fixed_sampler' and record.get('mode') == 'unconstrained'
]
benchmark_record = {
    'event': 'sampler_benchmarks',
    'mask_measurements': len(benchmark_mask_seconds),
    'mask_seconds_p50': percentile(benchmark_mask_seconds, 0.50),
    'mask_seconds_p95': percentile(benchmark_mask_seconds, 0.95),
    'mask_seconds_p99': percentile(benchmark_mask_seconds, 0.99),
    'mask_seconds_max': max(benchmark_mask_seconds),
    'mask_seconds_median': statistics.median(benchmark_mask_seconds),
    'vocabulary': sampler_vocabulary.stats.as_dict(),
    'benchmark_cache': benchmark_engine.cache_info(),
    'controller_caches': [
        controller.cache_info() if isinstance(controller, TriggeredMaskController)
        else controller.engine.cache_info()
        for controller in benchmark_controllers
    ],
    'constrained_tokens_per_second': [
        record['tokens_per_second'] for record in constrained_runs
    ],
    'unconstrained_tokens_per_second': [
        record['tokens_per_second'] for record in unconstrained_runs
    ],
}
JsonlRecorder(
    DRIVE_RECORDS / 'days-06-08-benchmarks.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append(benchmark_record)
print(json.dumps(benchmark_record, indent=2))


## Day 9: shim, structured responses, repair, and Qwen3.5

The endpoint separates three guarantees: native protocol parsing, JSON Schema validation, and semantic tool choice. `response_format=json_schema` renders the schema into the model-owned chat template and constrains the response with the same grammar IR. Validation still runs after constrained generation.

Non-streaming model-output failures receive one bounded repair request. Backend errors and truncation are never retried. Streaming remains incremental and cannot replay a second attempt after deltas have been sent.

Qwen3.5 uses `<function=...><parameter=...>` rather than Qwen3's JSON envelope. Its parser converts parameter values using the registered schema and emits JSON argument deltas. The Qwen3/b5401 runtime remains intact; the final section uses a separate cached llama.cpp b8233 build for Qwen3.5.


In [ ]:
import json
import random
from copy import deepcopy

from toolsmith.api import ChatCompletionRequest, ToolsmithService
from toolsmith.deltas import aggregate_chunks
from toolsmith.grammar import GBNFCompiler, ToolChoice
from toolsmith.parsing import Qwen3ToolCallParser
from toolsmith.protocols import ToolProtocol
from toolsmith.qwen35 import Qwen35IncrementalParser, Qwen35ToolCallParser
from toolsmith.recording import JsonlRecorder
from toolsmith.registry import Tool
from toolsmith.sampling.automaton import PushdownAutomaton
from toolsmith.sampling.mask import TokenMaskEngine
from toolsmith.sampling.vocabulary import Vocabulary, VocabularyEntry
from toolsmith.validation import (
    StructuredOutputValidator,
    ToolCallValidator,
)

DAY09_TOOLS = [
    Tool(
        'create_profile',
        'Create a profile',
        {
            'type': 'object',
            'properties': {
                'city': {'type': 'string'},
                'unit': {'enum': ['c', 'f']},
                'count': {'type': 'integer'},
                'ratio': {'type': 'number'},
                'active': {'type': 'boolean'},
                'nothing': {'type': 'null'},
                'tags': {'type': 'array', 'items': {'type': 'string'}},
                'location': {
                    'type': 'object',
                    'properties': {
                        'lat': {'type': 'number'},
                        'lon': {'type': 'number'},
                    },
                    'required': ['lat', 'lon'],
                    'additionalProperties': False,
                },
            },
            'required': ['city'],
            'additionalProperties': False,
        },
    ),
    Tool(
        'ping',
        'Ping',
        {'type': 'object', 'properties': {}, 'additionalProperties': False},
    ),
]
DAY09_FIXTURES = json.loads(
    (PROJECT_DIR / 'tests' / 'fixtures' / 'qwen35_cases.json').read_text()
)
day09_protocol_records = []
for fixture in DAY09_FIXTURES:
    parser = Qwen35ToolCallParser(
        DAY09_TOOLS,
        lambda: 'call_day09_fixture',
        expect_thinking=fixture['expect_thinking'],
    )
    expected = parser.parse(fixture['text'])
    incremental = Qwen35IncrementalParser(
        DAY09_TOOLS,
        lambda: 'call_day09_fixture',
        expect_thinking=fixture['expect_thinking'],
    )
    events = []
    randomizer = random.Random(23)
    offset = 0
    while offset < len(fixture['text']):
        size = randomizer.randint(1, 5)
        events.extend(incremental.feed(fixture['text'][offset:offset + size]))
        offset += size
    events.extend(incremental.finish())
    if incremental.response != expected:
        raise AssertionError(f"Qwen3.5 streaming mismatch: {fixture['name']}")
    ToolCallValidator(DAY09_TOOLS, ToolChoice('auto')).validate(expected)
    call = expected.tool_calls[0]
    if call.name != fixture['expected']['name']:
        raise AssertionError(f"Qwen3.5 tool name mismatch: {fixture['name']}")
    if json.loads(call.arguments) != fixture['expected']['arguments']:
        raise AssertionError(f"Qwen3.5 arguments mismatch: {fixture['name']}")
    day09_protocol_records.append(
        {
            'name': fixture['name'],
            'raw': fixture['text'],
            'parsed_tool_calls': [item.to_openai() for item in expected.tool_calls],
            'content': expected.content,
            'reasoning_content': expected.reasoning_content,
        }
    )

day09_compiler = GBNFCompiler(protocol=ToolProtocol.QWEN35)
day09_grammar = day09_compiler.compile_tools(DAY09_TOOLS, 'required')
day09_automaton = PushdownAutomaton.from_compiled(day09_grammar)
day09_valid_text = DAY09_FIXTURES[0]['text'].encode('utf-8')
day09_state = day09_automaton.initial_state()
for byte in day09_valid_text:
    day09_state = day09_automaton.advance_bytes(day09_state, bytes((byte,)))
    if day09_state is None:
        raise AssertionError('Qwen3.5 PDA rejected a valid prefix')
if not day09_automaton.can_finish(day09_state):
    raise AssertionError('Qwen3.5 PDA rejected a complete call')
for invalid in (
    '<tool_call>\n<function=ping>\n<parameter=extra>\nx\n</parameter>\n</function>\n</tool_call>',
    '<tool_call>\n<function=create_profile>\n</function>\n</tool_call>',
    DAY09_FIXTURES[0]['text'].replace('</parameter>', '', 1),
):
    state = day09_automaton.advance_bytes(day09_automaton.initial_state(), invalid.encode())
    if state is not None and day09_automaton.can_finish(state):
        raise AssertionError(f'Qwen3.5 PDA accepted invalid input: {invalid}')

day09_tokens = sorted({bytes((byte,)) for byte in day09_valid_text}) + [
    b'<tool_call>', b'</tool_call>', b'not-valid',
]
day09_vocabulary = Vocabulary(
    [VocabularyEntry(index, raw) for index, raw in enumerate(day09_tokens)],
    model_vocabulary_size=len(day09_tokens) + 1,
    tokenizer_vocabulary_size=len(day09_tokens),
    eos_token_ids=frozenset({len(day09_tokens)}),
    special_token_ids=frozenset({len(day09_tokens)}),
    excluded_empty_tokens=0,
    build_seconds=0,
)
day09_engine = TokenMaskEngine(day09_vocabulary, day09_automaton)
day09_mask_checks = 0
for offset in range(0, len(day09_valid_text) + 1, 9):
    state = day09_automaton.advance_bytes(
        day09_automaton.initial_state(), day09_valid_text[:offset]
    )
    if day09_engine.mask(state).allowed != day09_engine.brute_force_mask(state):
        raise AssertionError(f'Qwen3.5 trie/brute mask mismatch at {offset}')
    day09_mask_checks += 1

DAY09_RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {'answer': {'type': 'string'}, 'count': {'type': 'integer'}},
    'required': ['answer', 'count'],
    'additionalProperties': False,
}
StructuredOutputValidator(DAY09_RESPONSE_SCHEMA).validate(
    '{"answer":"sunny","count":2}'
)

class Day09Renderer:
    def __init__(self):
        self.calls = []
    def render(self, messages, tools=(), *, enable_thinking=None):
        self.calls.append((deepcopy(messages), deepcopy(tools), enable_thinking))
        return f'day09-prompt-{len(self.calls)}'

class Day09RepairClient:
    def __init__(self):
        self.calls = []
        self.responses = iter([
            {
                'content': '<tool_call>{"name":"get_weather","arguments":{"city":3}}</tool_call>',
                'tokens_evaluated': 10,
                'tokens_predicted': 4,
            },
            {
                'content': '<tool_call>{"name":"get_weather","arguments":{"city":"Paris"}}</tool_call>',
                'tokens_evaluated': 20,
                'tokens_predicted': 5,
            },
        ])
    def complete(self, prompt, **settings):
        self.calls.append((prompt, settings))
        return next(self.responses)

day09_repair_tool = Tool.from_openai({
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
            'additionalProperties': False,
        },
    },
})
day09_repair_client = Day09RepairClient()
day09_repair_renderer = Day09Renderer()
day09_repair_service = ToolsmithService(
    day09_repair_client,
    day09_repair_renderer,
    parser=Qwen3ToolCallParser(lambda: 'call_day09'),
)
day09_original_messages = [{'role': 'user', 'content': 'Call get_weather for Paris.'}]
day09_repaired = day09_repair_service.chat(ChatCompletionRequest(
    messages=day09_original_messages,
    tools=[day09_repair_tool.to_openai()],
    tool_choice='required',
    chat_template_kwargs={'enable_thinking': False},
    temperature=0.0,
))
if len(day09_repair_client.calls) != 2:
    raise AssertionError('Day 9 repair did not make exactly two attempts')
if day09_repaired['usage']['total_tokens'] != 39:
    raise AssertionError('Day 9 repair usage was not aggregated')
if day09_original_messages != [{'role': 'user', 'content': 'Call get_weather for Paris.'}]:
    raise AssertionError('Repair mutated the original request')
if day09_repair_client.calls[0][1]['grammar'] != day09_repair_client.calls[1][1]['grammar']:
    raise AssertionError('Repair dropped or changed the grammar')

qwen3_hash_checks = 0
if 'grammar_artifacts' in globals():
    for key, artifact in grammar_artifacts.items():
        if artifact['kind'] != 'tool':
            continue
        current = GBNFCompiler().compile_tools(grammar_tools, artifact['choice'])
        if current.sha256 != artifact['grammar_sha256']:
            raise AssertionError(f'Qwen3 grammar hash changed for {key}')
        qwen3_hash_checks += 1

day09_offline_record = {
    'event': 'day09_offline_replay',
    'protocol_cases': day09_protocol_records,
    'pda_valid_prefixes': len(day09_valid_text),
    'mask_checks': day09_mask_checks,
    'repair_attempts': len(day09_repair_client.calls),
    'repair_usage': day09_repaired['usage'],
    'qwen3_hash_checks': qwen3_hash_checks,
    'grammar_sha256': day09_grammar.sha256,
    'grammar_protocol': day09_grammar.protocol,
}
JsonlRecorder(
    DRIVE_RECORDS / 'day-09-offline.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append(day09_offline_record)
print(
    f"Day 9 offline replay: {len(day09_protocol_records)} protocol cases, "
    f"{day09_mask_checks} mask oracles, {len(day09_repair_client.calls)} repair attempts"
)


In [ ]:
from toolsmith.protocols import QWEN3_PROFILE
from toolsmith.repair import RepairPolicy

if not server.is_running:
    print('Restarting Qwen3-8B llama-server for the Day 9 API checks.', flush=True)
    server.start()
    health = server.wait_until_healthy(client, timeout=600, poll_interval=2)
    print(json.dumps(health, indent=2))

qwen3_models = openai_client.models.list()
if [model.id for model in qwen3_models.data] != [MODEL_NAME]:
    raise AssertionError(f'Unexpected Qwen3 model listing: {qwen3_models}')

DAY09_SDK_RESPONSE_FORMAT = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'weather_answer',
        'schema': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string'},
                'condition': {'type': 'string'},
                'temperature': {'type': 'number'},
            },
            'required': ['city', 'condition', 'temperature'],
            'additionalProperties': False,
        },
        'strict': True,
    },
}
qwen3_api_recorder = JsonlRecorder(
    DRIVE_RECORDS / 'day-09-qwen3-api.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
qwen3_structured = openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{
        'role': 'user',
        'content': 'Return JSON for Paris: sunny, 21.5 degrees.',
    }],
    response_format=DAY09_SDK_RESPONSE_FORMAT,
    max_tokens=192,
    temperature=0.0,
    seed=0,
    extra_body={'chat_template_kwargs': {'enable_thinking': False}},
)
qwen3_structured_content = qwen3_structured.choices[0].message.content
qwen3_structured_value = json.loads(qwen3_structured_content)
Draft202012Validator(DAY09_SDK_RESPONSE_FORMAT['json_schema']['schema']).validate(
    qwen3_structured_value
)
qwen3_api_recorder.append({
    'event': 'structured_response',
    'response': qwen3_structured.model_dump(),
    'decoded': qwen3_structured_value,
})

qwen3_structured_stream = openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{
        'role': 'user',
        'content': 'Return JSON for Paris: sunny, 21.5 degrees.',
    }],
    response_format=DAY09_SDK_RESPONSE_FORMAT,
    stream=True,
    stream_options={'include_usage': True},
    max_tokens=192,
    temperature=0.0,
    seed=0,
    extra_body={'chat_template_kwargs': {'enable_thinking': False}},
)
qwen3_structured_chunks = [chunk.model_dump() for chunk in qwen3_structured_stream]
qwen3_structured_aggregate = aggregate_chunks(qwen3_structured_chunks)
qwen3_stream_value = json.loads(qwen3_structured_aggregate['message']['content'])
Draft202012Validator(DAY09_SDK_RESPONSE_FORMAT['json_schema']['schema']).validate(
    qwen3_stream_value
)
if qwen3_structured_aggregate['finish_reason'] != 'stop':
    raise AssertionError('Structured stream did not finish normally')
qwen3_api_recorder.append({
    'event': 'structured_stream',
    'chunks': qwen3_structured_chunks,
    'aggregate': qwen3_structured_aggregate,
})

class InjectedQwen3Failure:
    def __init__(self, delegate):
        self.delegate = delegate
        self.calls = []
    def complete(self, prompt, **settings):
        self.calls.append((prompt, settings))
        if len(self.calls) == 1:
            return {
                'content': '<tool_call>{"name":"get_weather","arguments":{"city":3}}</tool_call>',
                'tokens_evaluated': 0,
                'tokens_predicted': 0,
            }
        return self.delegate.complete(prompt, **settings)

qwen3_injected_client = InjectedQwen3Failure(client)
qwen3_repair_recorder = JsonlRecorder(
    DRIVE_RECORDS / 'day-09-qwen3-repair.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
qwen3_repair_service = ToolsmithService(
    qwen3_injected_client,
    renderer,
    recorder=qwen3_repair_recorder,
    profile=QWEN3_PROFILE,
    repair_policy=RepairPolicy(max_repairs=1),
)
qwen3_repaired = qwen3_repair_service.chat(ChatCompletionRequest(
    model=MODEL_NAME,
    messages=[{
        'role': 'user',
        'content': 'Use get_weather exactly once for Paris in Celsius.',
    }],
    tools=grammar_tool_definitions,
    tool_choice='required',
    max_tokens=256,
    temperature=0.0,
    seed=0,
    chat_template_kwargs={'enable_thinking': False},
))
if len(qwen3_injected_client.calls) != 2:
    raise AssertionError('Injected Qwen3 failure did not result in one real retry')
qwen3_repaired_calls = qwen3_repaired['choices'][0]['message'].get('tool_calls', [])
if len(qwen3_repaired_calls) != 1 or qwen3_repaired_calls[0]['function']['name'] != 'get_weather':
    raise AssertionError(f'Qwen3 repair returned an unexpected call: {qwen3_repaired_calls}')
json.loads(qwen3_repaired_calls[0]['function']['arguments'])
qwen3_repair_recorder.append({
    'event': 'live_repair_result',
    'injected_first_failure': True,
    'attempts': len(qwen3_injected_client.calls),
    'response': qwen3_repaired,
})
print('Day 9 Qwen3 structured responses and injected-first-failure repair passed')


In [ ]:
import unicodedata

from toolsmith.prompting import normalize_history
from toolsmith.protocols import QWEN35_PROFILE, ToolProtocol
from toolsmith.qwen35 import Qwen35ToolCallParser

QWEN35_MODEL_NAME = QWEN35_PROFILE.model
QWEN35_GGUF_REPO = 'unsloth/Qwen3.5-9B-GGUF'
qwen35_config_path = hf_hub_download(
    repo_id=QWEN35_MODEL_NAME,
    filename='config.json',
    cache_dir='/content/hf-cache-qwen35-config',
    token=HF_TOKEN,
)
qwen35_config_json = json.loads(Path(qwen35_config_path).read_text())
qwen35_text_config = qwen35_config_json.get('text_config', qwen35_config_json)
QWEN35_MODEL_VOCAB_SIZE = qwen35_text_config.get('vocab_size')
if not isinstance(QWEN35_MODEL_VOCAB_SIZE, int):
    raise RuntimeError('Qwen3.5 config does not declare a text vocabulary size')
qwen35_tokenizer = AutoTokenizer.from_pretrained(
    QWEN35_MODEL_NAME,
    token=HF_TOKEN,
    use_fast=False,
)
qwen35_eos = qwen35_tokenizer.eos_token_id
qwen35_eos_ids = frozenset(
    qwen35_eos if isinstance(qwen35_eos, (list, tuple, set)) else {qwen35_eos}
)
qwen35_vocabulary = Vocabulary.from_qwen_tokenizer(
    qwen35_tokenizer,
    model_vocabulary_size=QWEN35_MODEL_VOCAB_SIZE,
    eos_token_ids=qwen35_eos_ids,
)
qwen35_vocabulary_audits = []
for text in (
    'plain café text 雪😀',
    '<tool_call>\n<function=get_weather>\n<parameter=city>\nZürich\n</parameter>\n</function>\n</tool_call>',
    '{"nested":[1,true,null]}',
):
    normalized = unicodedata.normalize('NFC', text)
    ids = qwen35_tokenizer(normalized, add_special_tokens=False)['input_ids']
    reconstructed = qwen35_vocabulary.reconstruct_bytes(ids)
    if reconstructed != normalized.encode('utf-8'):
        raise AssertionError('Qwen3.5 raw-byte vocabulary round trip failed')
    qwen35_vocabulary_audits.append({
        'text': text,
        'token_ids': ids,
        'raw_hex': reconstructed.hex(),
    })
for marker in ('<tool_call>', '</tool_call>', '<think>', '</think>'):
    token_id = qwen35_tokenizer.convert_tokens_to_ids(marker)
    if qwen35_vocabulary.raw_bytes(token_id) != marker.encode('utf-8'):
        raise AssertionError(f'Qwen3.5 marker byte mismatch: {marker}')

qwen35_renderer = ChatPromptRenderer(qwen35_tokenizer)
qwen35_prompt_snapshot = qwen35_renderer.render(
    [{'role': 'user', 'content': 'Call get_weather for Paris.'}],
    grammar_tool_definitions,
    enable_thinking=False,
)
if '<function=example_function_name>' not in qwen35_prompt_snapshot:
    raise AssertionError('Qwen3.5 official tool instructions were not rendered')
qwen35_history = [
    {'role': 'user', 'content': 'Weather?'},
    {
        'role': 'assistant',
        'content': None,
        'tool_calls': [{
            'id': 'call_snapshot',
            'type': 'function',
            'function': {
                'name': 'get_weather',
                'arguments': '{"city":"Paris","unit":"c"}',
            },
        }],
    },
    {'role': 'tool', 'tool_call_id': 'call_snapshot', 'content': 'sunny'},
]
qwen35_history_prompt = qwen35_renderer.render(
    normalize_history(qwen35_history, ToolProtocol.QWEN35),
    grammar_tool_definitions,
    enable_thinking=False,
)
if '<parameter=city>' not in qwen35_history_prompt or '<tool_response>' not in qwen35_history_prompt:
    raise AssertionError('Qwen3.5 native history rendering failed')

qwen35_protocol_compiler = GBNFCompiler(protocol=ToolProtocol.QWEN35)
qwen35_required_compiled = qwen35_protocol_compiler.compile_tools(
    grammar_tools, 'required'
)
qwen35_sample = (
    '<tool_call>\n<function=get_weather>\n<parameter=city>\nParis\n</parameter>\n'
    '<parameter=unit>\nc\n</parameter>\n</function>\n</tool_call>'
)
qwen35_sample_parsed = Qwen35ToolCallParser(
    grammar_tools, expect_thinking=False
).parse(qwen35_sample)
ToolCallValidator(grammar_tools, ToolChoice('required')).validate(qwen35_sample_parsed)
qwen35_sample_automaton = PushdownAutomaton.from_compiled(qwen35_required_compiled)
qwen35_sample_state = qwen35_sample_automaton.advance_bytes(
    qwen35_sample_automaton.initial_state(), qwen35_sample.encode()
)
if qwen35_sample_state is None or not qwen35_sample_automaton.can_finish(qwen35_sample_state):
    raise AssertionError('Qwen3.5 PDA rejected the native sample')
qwen35_grammar_path = DRIVE_GRAMMARS / 'day09-qwen35-required.gbnf'
qwen35_grammar_path.write_text(qwen35_required_compiled.text)
accepted, verdict = validate_gbnf(qwen35_grammar_path, qwen35_sample)
if not accepted:
    raise AssertionError(f'llama.cpp rejected Qwen3.5 native grammar sample: {verdict}')

qwen35_vocabulary_record = {
    'event': 'day09_qwen35_vocabulary',
    'model': QWEN35_MODEL_NAME,
    'model_vocab_size': QWEN35_MODEL_VOCAB_SIZE,
    'fingerprint': qwen35_vocabulary.fingerprint,
    'stats': qwen35_vocabulary.stats.as_dict(),
    'audits': qwen35_vocabulary_audits,
    'prompt_snapshot': qwen35_prompt_snapshot,
    'history_prompt': qwen35_history_prompt,
    'grammar_sha256': qwen35_required_compiled.sha256,
    'grammar_limitations': qwen35_required_compiled.limitations,
}
JsonlRecorder(
    DRIVE_RECORDS / 'day-09-qwen35-vocabulary.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append(qwen35_vocabulary_record)
print(json.dumps(qwen35_vocabulary_record['stats'], indent=2))


In [ ]:
QWEN35_LLAMA_REF = 'b8233'
QWEN35_LOCAL_SOURCE = Path('/content/llama.cpp-qwen35-source')
QWEN35_LOCAL_BUILD = Path('/content/llama.cpp-qwen35-build')
QWEN35_CACHED_BUILD = DRIVE_BUILD_CACHE / f'llama.cpp-{QWEN35_LLAMA_REF}-cuda-sm{GPU_COMPUTE_CAP}-runtime-v2'
QWEN35_SERVER_BIN = QWEN35_LOCAL_BUILD / 'bin' / 'llama-server'

def restore_qwen35_cached_build():
    def list_directory(path):
        try:
            return list(path.iterdir())
        except FileNotFoundError:
            return []
        except OSError as error:
            print(f'Could not list {path}: {error}', flush=True)
            return []

    # A fresh Drive mount hides children from stat until the directory is listed.
    list_directory(QWEN35_CACHED_BUILD.parent)
    list_directory(QWEN35_CACHED_BUILD)
    list_directory(QWEN35_CACHED_BUILD / 'bin')
    marker = QWEN35_CACHED_BUILD / '.complete'
    if marker.is_file() and marker.read_text() != QWEN35_LLAMA_REF + '\n':
        print(
            f'Ignoring {QWEN35_CACHED_BUILD}: completion marker does not match {QWEN35_LLAMA_REF}',
            flush=True,
        )
        return False
    binary = QWEN35_CACHED_BUILD / 'bin' / 'llama-server'
    if not binary.is_file():
        print(f'No cached llama-server at {binary}', flush=True)
        return False
    if not runtime_matches_gpu(QWEN35_CACHED_BUILD):
        return False
    staging = Path('/content/llama.cpp-qwen35-runtime-staging')
    remove_directory(staging)
    print(f'Restoring Qwen3.5 runtime from {QWEN35_CACHED_BUILD}', flush=True)
    shutil.copytree(QWEN35_CACHED_BUILD, staging, symlinks=True)
    if not server_works(staging):
        remove_directory(staging)
        print('Cached Qwen3.5 runtime is invalid; rebuilding locally.')
        return False
    remove_directory(QWEN35_LOCAL_BUILD)
    staging.rename(QWEN35_LOCAL_BUILD)
    return True

def cache_qwen35_build():
    staging = QWEN35_CACHED_BUILD.with_name(QWEN35_CACHED_BUILD.name + '-staging')
    remove_directory(staging)
    (staging / 'bin').mkdir(parents=True)
    shutil.copy2(QWEN35_SERVER_BIN, staging / 'bin' / 'llama-server')
    for artifact in (QWEN35_LOCAL_BUILD / 'bin').iterdir():
        if artifact.name.startswith('lib') and '.so' in artifact.name:
            if artifact.is_symlink():
                (staging / 'bin' / artifact.name).symlink_to(os.readlink(artifact))
            else:
                shutil.copy2(artifact, staging / 'bin' / artifact.name)
    if (QWEN35_LOCAL_BUILD / 'lib').is_dir():
        shutil.copytree(QWEN35_LOCAL_BUILD / 'lib', staging / 'lib', symlinks=True)
    (QWEN35_LOCAL_BUILD / '.compute-cap').write_text(GPU_COMPUTE_CAP + '\n')
    (staging / '.compute-cap').write_text(GPU_COMPUTE_CAP + '\n')
    (staging / '.complete').write_text(QWEN35_LLAMA_REF + '\n')
    remove_directory(QWEN35_CACHED_BUILD)
    staging.rename(QWEN35_CACHED_BUILD)

if server_works(QWEN35_LOCAL_BUILD) and runtime_matches_gpu(QWEN35_LOCAL_BUILD):
    print(f'Using validated local Qwen3.5 build: {QWEN35_SERVER_BIN}')
elif not restore_qwen35_cached_build():
    remove_directory(QWEN35_LOCAL_BUILD)
    if not (QWEN35_LOCAL_SOURCE / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--filter=blob:none', 'https://github.com/ggml-org/llama.cpp.git', str(QWEN35_LOCAL_SOURCE)],
            check=True,
        )
    subprocess.run(
        ['git', '-C', str(QWEN35_LOCAL_SOURCE), 'fetch', '--depth', '1', 'origin', 'tag', QWEN35_LLAMA_REF],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(QWEN35_LOCAL_SOURCE), 'checkout', '--detach', QWEN35_LLAMA_REF],
        check=True,
    )
    subprocess.run(
        [
            'cmake', '-S', str(QWEN35_LOCAL_SOURCE), '-B', str(QWEN35_LOCAL_BUILD),
            '-DGGML_CUDA=ON', f'-DCMAKE_CUDA_ARCHITECTURES={GPU_COMPUTE_CAP}',
            '-DCMAKE_BUILD_TYPE=Release', '-DLLAMA_CURL=ON',
        ],
        check=True,
    )
    subprocess.run(
        ['cmake', '--build', str(QWEN35_LOCAL_BUILD), '--target', 'llama-server', '-j', '8'],
        check=True,
    )
    if not server_works(QWEN35_LOCAL_BUILD):
        raise RuntimeError('Qwen3.5 llama-server failed runtime validation')
    cache_qwen35_build()
if not (server_works(QWEN35_LOCAL_BUILD) and runtime_matches_gpu(QWEN35_LOCAL_BUILD)):
    raise RuntimeError('No usable Qwen3.5 llama-server build for this GPU')

qwen35_model_info = HfApi(token=HF_TOKEN).model_info(
    QWEN35_GGUF_REPO, files_metadata=True
)
QWEN35_MODEL_REVISION = qwen35_model_info.sha
qwen35_candidates = [
    entry for entry in qwen35_model_info.siblings
    if entry.rfilename.endswith('.gguf')
    and 'Q4_K_M' in entry.rfilename
    and 'mmproj' not in entry.rfilename.lower()
]
if len(qwen35_candidates) != 1 or qwen35_candidates[0].size is None:
    raise RuntimeError(
        f'Expected exactly one sized Q4_K_M language GGUF, found: '
        f'{[entry.rfilename for entry in qwen35_candidates]}'
    )
qwen35_file_metadata = qwen35_candidates[0]
QWEN35_MODEL_FILE = qwen35_file_metadata.rfilename
QWEN35_EXPECTED_MODEL_SIZE = qwen35_file_metadata.size
QWEN35_LOCAL_MODEL = Path('/content') / f'qwen35-{Path(QWEN35_MODEL_FILE).name}'
QWEN35_DRIVE_MODEL_FILE = DRIVE_MODEL_CACHE / (
    f'{QWEN35_MODEL_REVISION[:12]}-{Path(QWEN35_MODEL_FILE).name}'
)
try:
    list(DRIVE_MODEL_CACHE.iterdir())
except OSError as error:
    print(f'Could not list {DRIVE_MODEL_CACHE}: {error}', flush=True)

def copy_qwen35_with_progress(source, destination):
    state = {'last_step': -1}
    def report(copied, total):
        percent = int(copied * 100 / total) if total else 100
        step = percent // 20
        if step != state['last_step']:
            print(f'{destination.name}: {percent}% ({copied:,}/{total:,} bytes)', flush=True)
            state['last_step'] = step
    return copy_file_atomic(
        source,
        destination,
        expected_size=QWEN35_EXPECTED_MODEL_SIZE,
        progress=report,
    )

if not is_gguf(QWEN35_LOCAL_MODEL, QWEN35_EXPECTED_MODEL_SIZE):
    if is_gguf(QWEN35_DRIVE_MODEL_FILE, QWEN35_EXPECTED_MODEL_SIZE):
        copy_qwen35_with_progress(QWEN35_DRIVE_MODEL_FILE, QWEN35_LOCAL_MODEL)
    else:
        downloaded = Path(hf_hub_download(
            repo_id=QWEN35_GGUF_REPO,
            filename=QWEN35_MODEL_FILE,
            revision=QWEN35_MODEL_REVISION,
            cache_dir='/content/hf-cache-qwen35',
            token=HF_TOKEN,
        ))
        if not is_gguf(downloaded, QWEN35_EXPECTED_MODEL_SIZE):
            downloaded = Path(hf_hub_download(
                repo_id=QWEN35_GGUF_REPO,
                filename=QWEN35_MODEL_FILE,
                revision=QWEN35_MODEL_REVISION,
                cache_dir='/content/hf-cache-qwen35',
                token=HF_TOKEN,
                force_download=True,
            ))
        if not is_gguf(downloaded, QWEN35_EXPECTED_MODEL_SIZE):
            raise RuntimeError('Downloaded Qwen3.5 GGUF failed validation')
        copy_qwen35_with_progress(downloaded, QWEN35_LOCAL_MODEL)
if not is_gguf(QWEN35_DRIVE_MODEL_FILE, QWEN35_EXPECTED_MODEL_SIZE):
    copy_qwen35_with_progress(QWEN35_LOCAL_MODEL, QWEN35_DRIVE_MODEL_FILE)
if not is_gguf(QWEN35_LOCAL_MODEL, QWEN35_EXPECTED_MODEL_SIZE):
    raise RuntimeError('Local Qwen3.5 GGUF failed final validation')
qwen35_runtime_metadata = {
    'llama_cpp_ref': QWEN35_LLAMA_REF,
    'model_repo': QWEN35_GGUF_REPO,
    'model_file': QWEN35_MODEL_FILE,
    'model_revision': QWEN35_MODEL_REVISION,
    'model_size': QWEN35_EXPECTED_MODEL_SIZE,
}
print(json.dumps(qwen35_runtime_metadata, indent=2))


In [ ]:
from toolsmith.backends import LlamaCppClient, LlamaCppServer
from toolsmith.repair import RepairPolicy

print('Finishing the HF sampler and Qwen3 phases before starting Qwen3.5.', flush=True)
for variable in ('sampler_model', 'sampler_tokenizer', 'sampler_config'):
    globals().pop(variable, None)
if 'torch' in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()
openai_client.close()
shim_server.should_exit = True
shim_thread.join(timeout=10)
close_listening_fds(SHIM_PORT)
client.close()
server.stop(timeout=20)
os.environ['LD_LIBRARY_PATH'] = server_environment(QWEN35_LOCAL_BUILD)['LD_LIBRARY_PATH']

qwen35_server = LlamaCppServer(
    executable=QWEN35_SERVER_BIN,
    model=QWEN35_LOCAL_MODEL,
    api_key=LOCAL_KEY,
    log_path=DRIVE_RECORDS / 'day-09-qwen35-server.log',
    host='127.0.0.1',
    port=SERVER_PORT,
    context_size=16384,
    gpu_layers=99,
)
qwen35_client = LlamaCppClient(qwen35_server.base_url, LOCAL_KEY, timeout=600.0)
qwen35_server.start()
qwen35_health = qwen35_server.wait_until_healthy(qwen35_client, timeout=900.0)
qwen35_service = ToolsmithService(
    qwen35_client,
    qwen35_renderer,
    profile=QWEN35_PROFILE,
    recorder=JsonlRecorder(
        DRIVE_RECORDS / 'day-09-qwen35-service.jsonl',
        secrets=(HF_TOKEN, LOCAL_KEY),
    ),
)
qwen35_app = create_app(qwen35_service, api_key=LOCAL_KEY)
qwen35_shim_server = uvicorn.Server(
    uvicorn.Config(
        qwen35_app, host='127.0.0.1', port=SHIM_PORT, log_level='warning',
        timeout_graceful_shutdown=5,
    )
)
qwen35_shim_thread = threading.Thread(target=qwen35_shim_server.run, daemon=True)
qwen35_shim_thread.start()
deadline = time.monotonic() + 30
while not qwen35_shim_server.started and time.monotonic() < deadline:
    if not qwen35_shim_thread.is_alive():
        raise RuntimeError('Qwen3.5 FastAPI shim stopped during startup')
    time.sleep(0.1)
if not qwen35_shim_server.started:
    raise TimeoutError('Qwen3.5 FastAPI shim did not start')
qwen35_openai_client = OpenAI(
    base_url=f'http://127.0.0.1:{SHIM_PORT}/v1',
    api_key=LOCAL_KEY,
    timeout=600.0,
    max_retries=0,
)
if [model.id for model in qwen35_openai_client.models.list().data] != [QWEN35_MODEL_NAME]:
    raise AssertionError('Qwen3.5 model listing mismatch')

qwen35_api_recorder = JsonlRecorder(
    DRIVE_RECORDS / 'day-09-qwen35-api.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
qwen35_api_recorder.append({
    'event': 'runtime_health',
    **qwen35_runtime_metadata,
    'health': qwen35_health,
    'context_size': 16384,
    'quality_scope': 'short non-thinking protocol checks, not long-context quality',
})
qwen35_raw_prompt = qwen35_renderer.render(
    [{'role': 'user', 'content': 'Use get_weather exactly once for Paris in Celsius.'}],
    grammar_tool_definitions,
    enable_thinking=False,
)
qwen35_raw_response = qwen35_client.complete(
    qwen35_raw_prompt,
    max_tokens=256,
    temperature=0.0,
    seed=0,
    grammar=qwen35_required_compiled.text,
)
qwen35_raw_parsed = Qwen35ToolCallParser(
    grammar_tools, expect_thinking=False
).parse(qwen35_raw_response.get('content', ''))
ToolCallValidator(grammar_tools, ToolChoice('required')).validate(qwen35_raw_parsed)
qwen35_api_recorder.append({
    'event': 'raw_native_completion',
    'prompt': qwen35_raw_prompt,
    'raw_response': qwen35_raw_response,
    'parsed': [call.to_openai() for call in qwen35_raw_parsed.tool_calls],
    'grammar_sha256': qwen35_required_compiled.sha256,
})

qwen35_sdk_results = {}
qwen35_sdk_cases = [
    ('auto-call', 'Use get_weather exactly once for Paris in Celsius.', 'auto', ['get_weather']),
    ('auto-text', 'What is 2 + 2? Answer directly without using a tool.', 'auto', []),
    ('none', 'Do not call a tool; explain that 2 + 2 is 4.', 'none', []),
    ('required', 'Call get_weather for Paris in Celsius.', 'required', ['get_weather']),
    (
        'named',
        'Find documentation about constrained decoding.',
        {'type': 'function', 'function': {'name': 'search_docs'}},
        ['search_docs'],
    ),
    (
        'parallel',
        'Make two calls: get_weather for Paris in Celsius and search_docs for constrained decoding.',
        'required',
        ['get_weather', 'search_docs'],
    ),
]
for label, prompt_text, choice, expected_names in qwen35_sdk_cases:
    response = qwen35_openai_client.chat.completions.create(
        model=QWEN35_MODEL_NAME,
        messages=[{'role': 'user', 'content': prompt_text}],
        tools=grammar_tool_definitions,
        tool_choice=choice,
        max_tokens=384,
        temperature=0.0,
        seed=0,
        extra_body={'chat_template_kwargs': {'enable_thinking': False}},
    )
    message = response.choices[0].message
    calls = message.tool_calls or []
    names = [call.function.name for call in calls]
    decoded = []
    for call in calls:
        arguments = json.loads(call.function.arguments)
        grammar_validators[call.function.name].validate(arguments)
        decoded.append({'name': call.function.name, 'arguments': arguments})
    semantic_match = (
        len(names) == 2 and set(names) == set(expected_names)
        if label == 'parallel'
        else names == expected_names
    )
    qwen35_api_recorder.append({
        'event': 'sdk_tool_choice',
        'label': label,
        'choice': choice,
        'response': response.model_dump(),
        'decoded_calls': decoded,
        'semantic_match': semantic_match,
        'false_call': not expected_names and bool(calls),
    })
    if not semantic_match:
        raise AssertionError(f'Qwen3.5 semantic tool choice failed for {label}: {names}')
    qwen35_sdk_results[label] = response

qwen35_stream = qwen35_openai_client.chat.completions.create(
    model=QWEN35_MODEL_NAME,
    messages=[{'role': 'user', 'content': 'Call get_weather for Paris in Celsius.'}],
    tools=grammar_tool_definitions,
    tool_choice='required',
    stream=True,
    stream_options={'include_usage': True},
    max_tokens=256,
    temperature=0.0,
    seed=0,
    extra_body={'chat_template_kwargs': {'enable_thinking': False}},
)
qwen35_stream_chunks = [chunk.model_dump() for chunk in qwen35_stream]
qwen35_stream_aggregate = aggregate_chunks(qwen35_stream_chunks)
qwen35_stream_calls = qwen35_stream_aggregate['message'].get('tool_calls', [])
if len(qwen35_stream_calls) != 1:
    raise AssertionError(f'Qwen3.5 stream call mismatch: {qwen35_stream_aggregate}')
json.loads(qwen35_stream_calls[0]['function']['arguments'])
qwen35_api_recorder.append({
    'event': 'sdk_stream',
    'chunks': qwen35_stream_chunks,
    'aggregate': qwen35_stream_aggregate,
})

qwen35_assistant = qwen35_sdk_results['required'].choices[0].message
qwen35_followup = qwen35_openai_client.chat.completions.create(
    model=QWEN35_MODEL_NAME,
    messages=[
        {'role': 'user', 'content': 'Call get_weather for Paris in Celsius.'},
        qwen35_assistant.model_dump(exclude_none=True),
        {
            'role': 'tool',
            'tool_call_id': qwen35_assistant.tool_calls[0].id,
            'content': json.dumps({'temperature': 21.5, 'condition': 'sunny'}),
        },
    ],
    tools=grammar_tool_definitions,
    tool_choice='auto',
    max_tokens=192,
    temperature=0.0,
    seed=0,
    extra_body={'chat_template_kwargs': {'enable_thinking': False}},
)
if not qwen35_followup.choices[0].message.content:
    raise AssertionError('Qwen3.5 tool-result follow-up was empty')
qwen35_api_recorder.append({
    'event': 'tool_result_followup',
    'response': qwen35_followup.model_dump(),
})

qwen35_structured = qwen35_openai_client.chat.completions.create(
    model=QWEN35_MODEL_NAME,
    messages=[{'role': 'user', 'content': 'Return JSON for Paris: sunny, 21.5 degrees.'}],
    response_format=DAY09_SDK_RESPONSE_FORMAT,
    max_tokens=192,
    temperature=0.0,
    seed=0,
    extra_body={'chat_template_kwargs': {'enable_thinking': False}},
)
qwen35_structured_value = json.loads(qwen35_structured.choices[0].message.content)
Draft202012Validator(DAY09_SDK_RESPONSE_FORMAT['json_schema']['schema']).validate(
    qwen35_structured_value
)
qwen35_api_recorder.append({
    'event': 'structured_response',
    'response': qwen35_structured.model_dump(),
    'decoded': qwen35_structured_value,
})

class InjectedQwen35Failure:
    def __init__(self, delegate):
        self.delegate = delegate
        self.calls = []
    def complete(self, prompt, **settings):
        self.calls.append((prompt, settings))
        if len(self.calls) == 1:
            return {
                'content': '<tool_call>\n<function=get_weather>\n</function>\n</tool_call>',
                'tokens_evaluated': 0,
                'tokens_predicted': 0,
            }
        return self.delegate.complete(prompt, **settings)

qwen35_injected_client = InjectedQwen35Failure(qwen35_client)
qwen35_repair_recorder = JsonlRecorder(
    DRIVE_RECORDS / 'day-09-qwen35-repair.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
)
qwen35_repair_service = ToolsmithService(
    qwen35_injected_client,
    qwen35_renderer,
    profile=QWEN35_PROFILE,
    recorder=qwen35_repair_recorder,
    repair_policy=RepairPolicy(max_repairs=1),
)
qwen35_repaired = qwen35_repair_service.chat(ChatCompletionRequest(
    model=QWEN35_MODEL_NAME,
    messages=[{'role': 'user', 'content': 'Call get_weather for Paris in Celsius.'}],
    tools=grammar_tool_definitions,
    tool_choice='required',
    max_tokens=256,
    temperature=0.0,
    seed=0,
    chat_template_kwargs={'enable_thinking': False},
))
if len(qwen35_injected_client.calls) != 2:
    raise AssertionError('Qwen3.5 injected failure did not make one real retry')
qwen35_repaired_calls = qwen35_repaired['choices'][0]['message'].get('tool_calls', [])
if len(qwen35_repaired_calls) != 1:
    raise AssertionError('Qwen3.5 repair did not return one call')
qwen35_repair_recorder.append({
    'event': 'live_repair_result',
    'injected_first_failure': True,
    'attempts': len(qwen35_injected_client.calls),
    'response': qwen35_repaired,
})

qwen35_thinking_service = ToolsmithService(
    qwen35_client,
    qwen35_renderer,
    profile=QWEN35_PROFILE,
    use_grammars=False,
    repair_policy=RepairPolicy(max_repairs=1),
)
qwen35_thinking_response = qwen35_thinking_service.chat(ChatCompletionRequest(
    model=QWEN35_MODEL_NAME,
    messages=[{
        'role': 'user',
        'content': 'Think briefly, then call get_weather exactly once for Paris in Celsius.',
    }],
    tools=grammar_tool_definitions,
    tool_choice='required',
    max_tokens=512,
    temperature=0.0,
    seed=0,
    chat_template_kwargs={'enable_thinking': True},
))
qwen35_thinking_message = qwen35_thinking_response['choices'][0]['message']
if not qwen35_thinking_message.get('tool_calls'):
    raise AssertionError('Qwen3.5 thinking prompt-only request returned no call')
qwen35_api_recorder.append({
    'event': 'thinking_prompt_only',
    'response': qwen35_thinking_response,
})
print('Day 9 Qwen3.5 native, streaming, structured, repair, and thinking checks passed')


In [ ]:
day09_result_record = {
    'event': 'day09_results',
    'models': [
        {
            'model': MODEL_NAME,
            'structured_schema_valid': True,
            'structured_stream_valid': True,
            'repair_attempts': len(qwen3_injected_client.calls),
            'repair_recovered': bool(qwen3_repaired_calls),
            'repair_usage': qwen3_repaired['usage'],
        },
        {
            'model': QWEN35_MODEL_NAME,
            **qwen35_runtime_metadata,
            'native_cases': len(qwen35_sdk_results),
            'native_stream_calls': len(qwen35_stream_calls),
            'structured_schema_valid': True,
            'repair_attempts': len(qwen35_injected_client.calls),
            'repair_recovered': bool(qwen35_repaired_calls),
            'thinking_tool_calls': len(qwen35_thinking_message.get('tool_calls', [])),
            'grammar_sha256': qwen35_required_compiled.sha256,
        },
    ],
    'qwen3_ablation': [
        {
            'case': record['case'],
            'mode': record['mode'],
            'parse_rate': record['parse_rate'],
            'schema_valid': record['schema_valid'],
            'semantic_choice': record['choice_correct'],
            'false_call': record['false_call'],
            'tokens_per_second': record['tokens_per_second'],
        }
        for record in ablation_results
    ],
}
JsonlRecorder(
    DRIVE_RECORDS / 'day-09-results.jsonl',
    secrets=(HF_TOKEN, LOCAL_KEY),
).append(day09_result_record)
print(json.dumps(day09_result_record, indent=2))


## Day 10: reproducible evaluation

These cells evaluate the 200-case gold set, BFCL V4 Python single-turn AST categories, and JSONSchemaBench/MaskBench. Full BFCL scores refer to prompt/thinking baselines over every ID. Constrained scores use exact eligible subsets because unsupported schemas are rejected, never dropped silently. JSONSchemaBench generation uses 25 schemas per split; MaskBench uses 100 per split, while compile coverage attempts every schema. All scores are produced by harnesses and recorded on Drive, never hardcoded.


In [ ]:
import json
from toolsmith.evaluation.bfcl_adapter import constraint_support
from toolsmith.evaluation.bfcl_handler import decode_standardized_result
from toolsmith.evaluation.datasets import load_cases, load_tool_definitions, validate_toolsmith_dataset
from toolsmith.evaluation.metrics import calls_equal, deterministic_subset
from toolsmith.evaluation.report import markdown_summary, summarize_by_group
from toolsmith.evaluation.runner import atomic_write_json

DAY10_CASES = load_cases(PROJECT_DIR / 'evaluation/data/toolsmith_single_turn_v1.jsonl')
DAY10_TOOLS = load_tool_definitions(PROJECT_DIR / 'evaluation/data/toolsmith_tools_v1.json')
day10_dataset_summary = validate_toolsmith_dataset(DAY10_CASES, DAY10_TOOLS)
if day10_dataset_summary['cases'] != 200 or day10_dataset_summary['tools'] != 10:
    raise AssertionError(day10_dataset_summary)
selected = deterministic_subset(DAY10_CASES, 12, seed=10, id_key='case_id')
if len(selected) != 12 or selected != deterministic_subset(DAY10_CASES, 12, seed=10, id_key='case_id'):
    raise AssertionError('Day 10 subset selection is not deterministic')
if not calls_equal(selected[0]['expected_calls'], selected[0]['expected_calls'], selected[0]['comparison']):
    raise AssertionError('Gold call comparison failed')
unsupported = constraint_support([{
    'type': 'function',
    'function': {
        'name': 'bad',
        'parameters': {
            'type': 'object',
            'properties': {'x': {'type': 'string', 'pattern': '^x$'}},
            'additionalProperties': False,
        },
    },
}])
if unsupported.eligible or unsupported.error['pointer'] != '/properties/x/pattern':
    raise AssertionError('Unsupported schema was not surfaced with its pointer')
if decode_standardized_result([{'f': '{"x":1}'}]) != [{'f': {'x': 1}}]:
    raise AssertionError('BFCL result decoder failed')
report_text = markdown_summary(summarize_by_group([]), title='Day 10 offline validation')
atomic_write_json(
    Path(DRIVE_RECORDS) / 'day-10-offline-summary.json',
    {'dataset': day10_dataset_summary, 'smoke_case_ids': [case['case_id'] for case in selected]},
)
print(json.dumps(day10_dataset_summary, indent=2))


In [ ]:
import re
import shutil
import site
import tomllib

DAY10_LOCK = json.loads((PROJECT_DIR / 'evaluation/locks/day10.json').read_text())
DAY10_EVAL_ROOT = Path('/content/toolsmith-evaluation')
DAY10_EVAL_ROOT.mkdir(parents=True, exist_ok=True)
# Set before any bfcl_eval import: it creates result/score/lock dirs under this root.
os.environ['BFCL_PROJECT_ROOT'] = str(DRIVE_RECORDS / 'day-10')
os.environ['HF_DATASETS_CACHE'] = str(DRIVE_MODEL_CACHE / 'datasets')

def run_checked(command, **kwargs):
    # Subprocess output never reaches the notebook, so surface it on failure.
    command = [str(part) for part in command]
    result = subprocess.run(command, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        output = (result.stdout + result.stderr)[-8000:]
        raise RuntimeError(f'Command failed with exit {result.returncode}: {" ".join(command)}\n{output}')
    return result

def checkout_pinned(name, lock):
    destination = DAY10_EVAL_ROOT / name
    if not (destination / '.git').exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', lock['repository'], str(destination)], check=True)
    if subprocess.run(['git', '-C', str(destination), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout.strip():
        raise RuntimeError(f'Pinned evaluation checkout has local edits: {destination}')
    subprocess.run(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', lock['commit']], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', lock['commit']], check=True)
    actual = subprocess.run(['git', '-C', str(destination), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
    if actual != lock['commit']:
        raise AssertionError(f'{name} revision mismatch: {actual}')
    return destination

BFCL_SOURCE = checkout_pinned('gorilla', DAY10_LOCK['bfcl'])
JSONSCHEMA_CORE_SOURCE = checkout_pinned('jsonschemabench-core', DAY10_LOCK['jsonschemabench_core'])
MASKBENCH_SOURCE = checkout_pinned('jsonschemabench-maskbench', DAY10_LOCK['maskbench'])
MASKBENCH_PROJECT = MASKBENCH_SOURCE / 'maskbench'  # the `maskbench` package lives in maskbench/maskbench

# BFCL runs inside this Python 3.13 kernel. Colab's system Python has no ensurepip,
# so the venv reuses the system pip through --system-site-packages.
EVAL_VENV = DAY10_EVAL_ROOT / 'venv'
EVAL_PYTHON = EVAL_VENV / 'bin/python'
if not EVAL_PYTHON.exists():
    run_checked([sys.executable, '-m', 'venv', '--clear', '--without-pip', '--system-site-packages', EVAL_VENV])
run_checked([EVAL_PYTHON, '-m', 'pip', '--version'])
bfcl_project = BFCL_SOURCE / 'berkeley-function-call-leaderboard'
bfcl_requirements = tomllib.loads((bfcl_project / 'pyproject.toml').read_text())['project']['dependencies']
# numpy==1.26.4 has no Python 3.13 wheel and cannot build on 3.13; use the
# runtime's numpy 2, which this kernel has already imported anyway.
bfcl_requirements = ['numpy>=2' if re.match(r'numpy\s*==', item) else item for item in bfcl_requirements]
run_checked([EVAL_PYTHON, '-m', 'pip', 'install', '-q', '--no-deps', '-e', bfcl_project])
run_checked([
    EVAL_PYTHON, '-m', 'pip', 'install', '-q', *bfcl_requirements,
    'datasets>=2.19,<5', 'stopit', 'overrides', 'matplotlib',
])
run_checked([EVAL_PYTHON, '-c', 'import bfcl_eval.utils, bfcl_eval.model_handler.base_handler, bfcl_eval.eval_checker.eval_runner'])
eval_site = run_checked(
    [EVAL_PYTHON, '-c', 'import site; print(site.getsitepackages()[0])'],
).stdout.strip()
site.addsitedir(eval_site)
for source in (JSONSCHEMA_CORE_SOURCE, MASKBENCH_PROJECT):
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))

# MaskBench peer engines run as subprocesses in their own uv-managed Python 3.12
# venv, because outlines_core 0.2.3 has no Python 3.13 wheel. grid-guardrail
# 0.2.5 was never published to PyPI, so pin 0.3.0, the nearest release, here
# rather than relying on the checkout's lock file being up to date.
PEER_VERSIONS = {**DAY10_LOCK['peer_versions'], 'grid-guardrail': '0.3.0'}
if shutil.which('uv') is None:
    run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv') or str(Path(sys.executable).parent / 'uv')
PEER_VENV = DAY10_EVAL_ROOT / 'peer-venv-py312'
PEER_PYTHON = PEER_VENV / 'bin/python'
if not PEER_PYTHON.exists():
    run_checked([UV, 'venv', '--clear', '--python', '3.12', PEER_VENV])
peer_overrides = DAY10_EVAL_ROOT / 'peer-overrides.txt'
peer_overrides.write_text(f"grid-guardrail=={PEER_VERSIONS['grid-guardrail']}\n")
peer_build_env = {**os.environ, 'CMAKE_BUILD_PARALLEL_LEVEL': str(os.cpu_count() or 2)}
peer_build_env.pop('VIRTUAL_ENV', None)
# Peer mask computation is CPU-only, so skip the multi-GB CUDA torch wheels.
run_checked([UV, 'pip', 'install', '--python', PEER_PYTHON, 'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'], env=peer_build_env)
run_checked([
    UV, 'pip', 'install', '--python', PEER_PYTHON,
    '-r', MASKBENCH_PROJECT / 'requirements.txt', '--override', peer_overrides,
], env=peer_build_env)
PEER_ENV = {**os.environ, 'PYTHONPATH': str(MASKBENCH_PROJECT)}
PEER_ENV.pop('VIRTUAL_ENV', None)
peer_installed = json.loads(run_checked([
    PEER_PYTHON, '-c',
    'import importlib.metadata as m, json, sys; print(json.dumps({n: m.version(n) for n in sys.argv[1:]}))',
    *PEER_VERSIONS,
], env=PEER_ENV).stdout)
if peer_installed != PEER_VERSIONS:
    raise AssertionError(f'Peer engine versions differ from the lock: {peer_installed}')
run_checked([PEER_PYTHON, '-c', 'import maskbench.runner, llguidance, xgrammar, llama_cpp, outlines_core, grid'], env=PEER_ENV)
print(f'Evaluation environment: {EVAL_PYTHON}; BFCL {BFCL_SOURCE}')
print(f'MaskBench peer environment: {PEER_PYTHON}; {peer_installed}')

In [ ]:
import importlib

import toolsmith.backends as _backends
import toolsmith.backends.llama_cpp_server as _server_mod
import toolsmith.evaluation.bfcl_runner as _bfcl_mod
import toolsmith.evaluation.jsonschema_runner as _json_mod
import toolsmith.evaluation.own_runner as _own_mod
import toolsmith.evaluation.runner as _runner_mod

importlib.reload(_server_mod)
importlib.reload(_backends)
importlib.reload(_runner_mod)
importlib.reload(_own_mod)
importlib.reload(_bfcl_mod)
importlib.reload(_json_mod)
LlamaCppServer = _backends.LlamaCppServer
LlamaCppClient = _backends.LlamaCppClient

from toolsmith.evaluation.manifest import build_manifest
from toolsmith.evaluation.models import Mode, RunConfig, canonical_hash
from toolsmith.evaluation.own_runner import OwnRunner, run_repair_with_reuse
from toolsmith.evaluation.policies import service_options
from toolsmith.protocols import QWEN3_PROFILE, QWEN35_PROFILE

if 'stop_day10_runtime' in globals():
    stop_day10_runtime()

DAY10_ACTIVE = {}
DAY10_OWN_RESULTS = {}
DAY10_BFCL_RESULTS = {}
DAY10_RECORDS = Path(DRIVE_RECORDS) / 'day-10'
DAY10_RECORDS.mkdir(parents=True, exist_ok=True)

def stop_day10_runtime():
    global DAY10_ACTIVE
    for group in DAY10_ACTIVE.get('mode_clients', {}).values():
        for client in group.values():
            client.close()
    if DAY10_ACTIVE.get('native_client') is not None:
        DAY10_ACTIVE['native_client'].close()
    for shim in DAY10_ACTIVE.get('shims', []):
        shim.should_exit = True
    for thread in DAY10_ACTIVE.get('threads', []):
        thread.join(timeout=10)
    if DAY10_ACTIVE.get('client') is not None:
        DAY10_ACTIVE['client'].close()
    if DAY10_ACTIVE.get('server') is not None:
        DAY10_ACTIVE['server'].stop(timeout=20)
    DAY10_ACTIVE = {}
    for name in ('qwen35_openai_client', 'openai_client'):
        if name in globals():
            globals()[name].close()
    for server_name, thread_name in (('qwen35_shim_server', 'qwen35_shim_thread'), ('shim_server', 'shim_thread')):
        if server_name in globals():
            globals()[server_name].should_exit = True
            if thread_name in globals():
                globals()[thread_name].join(timeout=10)
    if 'close_listening_fds' in globals() and 'SHIM_PORT' in globals():
        for offset in range(8):
            close_listening_fds(SHIM_PORT + offset)
    for name in ('qwen35_client', 'client'):
        if name in globals():
            globals()[name].close()
    for name in ('qwen35_server', 'server'):
        if name in globals():
            globals()[name].stop(timeout=20)

def stop_listening_llama_servers():
    import signal
    for name in ('server', 'qwen35_server'):
        obj = globals().get(name)
        if obj is not None and getattr(obj, 'is_running', False):
            obj.stop(timeout=20)
    try:
        tcp_lines = Path('/proc/net/tcp').read_text().splitlines()[1:]
    except OSError:
        return
    listening_inodes = set()
    for line in tcp_lines:
        fields = line.split()
        if fields[3] == '0A':
            listening_inodes.add(fields[9])
    if not listening_inodes:
        return
    pids = set()
    for proc in Path('/proc').iterdir():
        if not proc.name.isdigit():
            continue
        try:
            command = (proc / 'cmdline').read_bytes().replace(b'\x00', b' ').decode(errors='replace')
            if 'llama-server' not in command:
                continue
            owns_listener = any(
                os.readlink(fd).startswith('socket:[') and os.readlink(fd)[8:-1] in listening_inodes
                for fd in (proc / 'fd').iterdir()
            )
        except OSError:
            continue
        if owns_listener:
            pids.add(int(proc.name))
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except OSError:
            pass
    deadline = time.monotonic() + 10
    while pids and time.monotonic() < deadline:
        pids = {pid for pid in pids if Path(f'/proc/{pid}').exists()}
        time.sleep(0.1)
    for pid in pids:
        try:
            os.kill(pid, signal.SIGKILL)
        except OSError:
            pass

def start_day10_runtime(profile):
    global DAY10_ACTIVE, SERVER_PORT
    running = DAY10_ACTIVE.get('server')
    if (
        DAY10_ACTIVE.get('profile') == profile
        and getattr(running, 'parallel', 1) == 4
        and getattr(running, 'is_running', False)
    ):
        return DAY10_ACTIVE
    stop_day10_runtime()
    stop_listening_llama_servers()
    with socket.socket() as probe:
        try:
            probe.bind(('127.0.0.1', SERVER_PORT))
        except OSError:
            SERVER_PORT = first_free_port('127.0.0.1', 8080)
            print(f'Day 10 llama-server will listen on {SERVER_PORT}', flush=True)
    if profile == QWEN3_PROFILE:
        executable, model_path, build = SERVER_BIN, LOCAL_MODEL, LOCAL_BUILD
        active_renderer = renderer
    else:
        executable, model_path, build = QWEN35_SERVER_BIN, QWEN35_LOCAL_MODEL, QWEN35_LOCAL_BUILD
        active_renderer = qwen35_renderer
    os.environ['LD_LIBRARY_PATH'] = server_environment(build)['LD_LIBRARY_PATH']
    server = LlamaCppServer(
        executable=executable, model=model_path, api_key=LOCAL_KEY,
        log_path=DAY10_RECORDS / f'{profile.name}-server.log',
        host='127.0.0.1', port=SERVER_PORT, context_size=16384, gpu_layers=99,
        parallel=4,
    )
    client = LlamaCppClient(server.base_url, LOCAL_KEY, timeout=900.0)
    server.start()
    health = server.wait_until_healthy(client, timeout=900.0)
    mode_clients, shims, threads = {'own': {}, 'bfcl': {}}, [], []
    audiences = (('own', False), ('bfcl', True))
    slot = 0
    for mode in (Mode.PROMPT, Mode.GRAMMAR, Mode.REPAIR, Mode.THINKING):
      for audience, defer_schema in audiences:
        options = service_options(mode, defer_bfcl_schema=defer_schema)
        service = ToolsmithService(
            client, active_renderer, profile=profile,
            recorder=JsonlRecorder(
                DAY10_RECORDS / f'{profile.name}-{audience}-{mode.value}-service.jsonl',
                secrets=(HF_TOKEN, LOCAL_KEY),
            ),
            **options,
        )
        shim = uvicorn.Server(uvicorn.Config(
            create_app(service, api_key=LOCAL_KEY),
            host='127.0.0.1', port=SHIM_PORT + slot, log_level='warning',
            timeout_graceful_shutdown=5,
        ))
        thread = threading.Thread(target=shim.run, daemon=True)
        thread.start()
        deadline = time.monotonic() + 30
        while not shim.started and time.monotonic() < deadline:
            if not thread.is_alive():
                raise RuntimeError(f'{mode.value} shim failed to start')
            time.sleep(0.1)
        if not shim.started:
            raise TimeoutError(f'{mode.value} shim startup timed out')
        mode_clients[audience][mode] = OpenAI(
            base_url=f'http://127.0.0.1:{SHIM_PORT + slot}/v1',
            api_key=LOCAL_KEY, timeout=900.0, max_retries=0,
        )
        shims.append(shim)
        threads.append(thread)
        slot += 1
    DAY10_ACTIVE = {
        'profile': profile, 'server': server, 'client': client,
        'renderer': active_renderer, 'mode_clients': mode_clients,
        'native_client': OpenAI(base_url=server.base_url + '/v1', api_key=LOCAL_KEY, timeout=900.0, max_retries=0),
        'shims': shims, 'threads': threads, 'health': health,
    }
    print(f'Started Day 10 runtime: {profile.model}')
    return DAY10_ACTIVE

def run_day10_own(profile):
    active = start_day10_runtime(profile)
    outputs = {}
    for mode in (Mode.PROMPT, Mode.GRAMMAR, Mode.THINKING):
        config = RunConfig(f'own-{profile.name}-{mode.value}', 'own-v1', profile.model, mode)
        manifest = canonical_hash({'config': config.as_dict(), 'dataset': day10_dataset_summary})
        runner = OwnRunner(active['mode_clients']['own'][mode], config, DAY10_TOOLS)
        outputs[mode.value] = runner.run(
            DAY10_CASES, DAY10_RECORDS / f'own-{profile.name}-{mode.value}.jsonl', manifest,
        )
    grammar = outputs[Mode.GRAMMAR.value]
    repair_config = RunConfig(f'own-{profile.name}-repair', 'own-v1', profile.model, Mode.REPAIR)
    outputs[Mode.REPAIR.value] = run_repair_with_reuse(
        grammar, DAY10_CASES,
        OwnRunner(active['mode_clients']['own'][Mode.REPAIR], repair_config, DAY10_TOOLS),
        DAY10_RECORDS / f'own-{profile.name}-repair.jsonl',
        canonical_hash({'config': repair_config.as_dict(), 'dataset': day10_dataset_summary}),
    )
    DAY10_OWN_RESULTS[profile.name] = outputs
    return outputs


In [ ]:
print(json.dumps(day10_dataset_summary, indent=2))
print('Smoke case order:', [case['case_id'] for case in selected])
print('Long runs are resumable and will not mix results when the manifest changes.')


In [ ]:
day10_qwen3_own = run_day10_own(QWEN3_PROFILE)
print({mode: len(results) for mode, results in day10_qwen3_own.items()})


In [ ]:
day10_qwen35_own = run_day10_own(QWEN35_PROFILE)
print({mode: len(results) for mode, results in day10_qwen35_own.items()})


In [ ]:
import importlib

import toolsmith.evaluation.bfcl_runner as _bfcl_mod
importlib.reload(_bfcl_mod)
from toolsmith.evaluation.bfcl_adapter import BFCL_CATEGORIES, convert_bfcl_tools, eligible_ids
from toolsmith.evaluation.bfcl_runner import BFCLRunner, run_bfcl_repair_with_reuse

def load_bfcl_entries(category, limit=None):
    from bfcl_eval.utils import load_dataset_entry
    entries = load_dataset_entry(category, include_prereq=False, include_language_specific_hint=False)
    return entries if limit is None else entries[:limit]

def run_day10_bfcl(profile, limit=None):
    active = start_day10_runtime(profile)
    # Checkpoints are stamped with the entry IDs they cover, so a pilot must not
    # share files with the full run or the full run raises ManifestMismatchError.
    suffix = '' if limit is None else f'-pilot{limit}'
    outputs = {}
    for category in BFCL_CATEGORIES:
        entries = load_bfcl_entries(category, limit)
        for mode in (Mode.PROMPT, Mode.GRAMMAR, Mode.THINKING):
            config = RunConfig(f'bfcl-{profile.name}-{category}-{mode.value}', 'bfcl-v4', profile.model, mode)
            manifest = canonical_hash({'config': config.as_dict(), 'ids': [entry['id'] for entry in entries]})
            runner = BFCLRunner(
                active['mode_clients']['bfcl'][mode], config,
                protocol=profile.protocol, tools_converter=convert_bfcl_tools,
            )
            results = runner.run(entries, DAY10_RECORDS / f'bfcl-{profile.name}-{category}-{mode.value}{suffix}.jsonl', manifest)
            outputs[(category, mode.value)] = results
        grammar = outputs[(category, Mode.GRAMMAR.value)]
        config = RunConfig(f'bfcl-{profile.name}-{category}-repair', 'bfcl-v4', profile.model, Mode.REPAIR)
        outputs[(category, Mode.REPAIR.value)] = run_bfcl_repair_with_reuse(
            grammar, entries,
            BFCLRunner(
                active['mode_clients']['bfcl'][Mode.REPAIR], config,
                protocol=profile.protocol, tools_converter=convert_bfcl_tools,
            ),
            DAY10_RECORDS / f'bfcl-{profile.name}-{category}-repair{suffix}.jsonl',
            canonical_hash({'config': config.as_dict(), 'ids': [entry['id'] for entry in entries]}),
        )
    DAY10_BFCL_RESULTS[profile.name] = outputs
    return outputs

day10_bfcl_pilot_qwen3 = run_day10_bfcl(QWEN3_PROFILE, limit=2)
day10_bfcl_pilot_qwen35 = run_day10_bfcl(QWEN35_PROFILE, limit=2)
print('BFCL pilot complete; full runs still required for full-score claims.')


In [ ]:
day10_qwen3_bfcl = run_day10_bfcl(QWEN3_PROFILE)
print('Qwen3 BFCL generation complete')


In [ ]:
day10_qwen35_bfcl = run_day10_bfcl(QWEN35_PROFILE)
print('Qwen3.5 BFCL generation complete')


In [ ]:
import importlib

import toolsmith.evaluation.bfcl_runner as _bfcl_mod
importlib.reload(_bfcl_mod)
from toolsmith.evaluation.bfcl_handler import register_handler
from toolsmith.evaluation.bfcl_runner import evaluate_bfcl, score_entries

bfcl_result_root = DAY10_RECORDS / 'bfcl-result'
bfcl_score_root = DAY10_RECORDS / 'bfcl-score'
bfcl_eligible = {}
for profile_name, results_by_key in DAY10_BFCL_RESULTS.items():
    profile = QWEN3_PROFILE if profile_name == 'qwen3' else QWEN35_PROFILE
    for (category, mode), results in results_by_key.items():
        registry = f'toolsmith-{profile_name}-{mode}'
        full_category = mode not in {'grammar', 'repair'}
        entries = score_entries(results, full=full_category)
        bfcl_eligible[f'{profile_name}/{category}/{mode}'] = sorted(eligible_ids(results))
        config = register_handler(registry, model_name=profile.model)
        handler = config.model_handler(
            model_name=profile.model, temperature=0,
            registry_name=registry, is_fc_model=True,
        )
        handler.write(entries, bfcl_result_root, update_mode=True)
        evaluate_bfcl(
            registry_name=registry, model_name=profile.model,
            categories=[category], result_dir=str(bfcl_result_root),
            score_dir=str(bfcl_score_root),
            partial=not full_category,
        )
atomic_write_json(DAY10_RECORDS / 'bfcl-eligible-ids.json', bfcl_eligible)
print(f'BFCL scores: {bfcl_score_root}')


Number of models evaluated: 100%|██████████| 2/2 [00:00<00:00, 53.58it/s]


🦍 Model: toolsmith-qwen3-prompt
🔍 Running test: simple_python
✅ Test completed: simple_python. 🎯 Accuracy: 95.25%
📈 Aggregating data to generate leaderboard score table...
🏁 Evaluation completed. See /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_overall.csv for overall evaluation results on BFCL V4.
See /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_live.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_non_live.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_multi_turn.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_agentic.csv and /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_format_sensitivity.csv for detailed evaluation results on each sub-section categories respectively.


Number of models evaluated: 100%|██████████| 2/2 [00:00<00:00, 13957.75it/s]

📈 Aggregating data to generate leaderboard score table...


🏁 Evaluation completed. See /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_overall.csv for overall evaluation results on BFCL V4.
⚠️  Partial evaluation for a single category is enabled (--partial-run flag is set). Accuracy scores are computed only on the subset of entries present in the model result files, which may differ from a full evaluation and from the official leaderboard score.
See /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_live.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_non_live.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_multi_turn.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_agentic.csv and /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_format_sensitivity.csv for detailed evaluation results on each sub-section categories respectively.


Number of models evaluated:   0%|          | 0/2 [00:00<?, ?it/s]

... (110 outputs left out)

📈 Aggregating data to generate leaderboard score table...
🏁 Evaluation completed. See /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_overall.csv for overall evaluation results on BFCL V4.
See /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_live.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_non_live.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_multi_turn.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_agentic.csv and /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_format_sensitivity.csv for detailed evaluation results on each sub-section categories respectively.


Number of models evaluated: 100%|██████████| 4/4 [00:00<00:00, 21620.12it/s]

📈 Aggregating data to generate leaderboard score table...


🏁 Evaluation completed. See /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_overall.csv for overall evaluation results on BFCL V4.
⚠️  Partial evaluation for a single category is enabled (--partial-run flag is set). Accuracy scores are computed only on the subset of entries present in the model result files, which may differ from a full evaluation and from the official leaderboard score.
See /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_live.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_non_live.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_multi_turn.csv, /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_agentic.csv and /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score/data_format_sensitivity.csv for detailed evaluation results on each sub-section categories respectively.
BFCL scores: /content/drive/MyDrive/toolsmith/records/day-10/bfcl-score


In [ ]:
from collections import Counter
from toolsmith.evaluation.metrics import deterministic_subset
from toolsmith.evaluation.schema_coverage import isolated_compile_coverage, replay_schema

maskbench_files_by_split = {}
for file in sorted((MASKBENCH_SOURCE / 'maskbench/data').glob('*.json')):
    split = file.name.split('---')[0]
    maskbench_files_by_split.setdefault(split, []).append(file)
day10_maskbench_files = []
coverage_recorder = JsonlRecorder(DAY10_RECORDS / 'schema-coverage.jsonl', secrets=(HF_TOKEN, LOCAL_KEY))
for split, files in sorted(maskbench_files_by_split.items()):
    selected_files = deterministic_subset(files, 100, seed=10, id_key='name')
    day10_maskbench_files.extend(selected_files)
    for file in files:
        schema_data = json.loads(file.read_text())
        coverage_recorder.append(isolated_compile_coverage(schema_data['schema'], file.name))
    for file in selected_files:
        schema_data = json.loads(file.read_text())
        coverage_recorder.append({
            'event': 'instance_replay',
            **replay_schema(schema_data['schema'], schema_data['tests'], file.name),
        })
print(f'Coverage complete; selected {len(day10_maskbench_files)} schemas for MaskBench')


In [ ]:
import importlib
from functools import partial
from datasets import load_dataset
import toolsmith.evaluation.jsonschema_runner as _json_mod
importlib.reload(_json_mod)
from toolsmith.evaluation.jsonschema_engine import create_engine_class
from toolsmith.evaluation.jsonschema_runner import run_selected_generation

JSONSCHEMA_SPLITS = [
    'Github_easy', 'Github_hard', 'Github_medium', 'Github_trivial', 'Github_ultra',
    'Glaiveai2K', 'JsonSchemaStore', 'Kubernetes', 'Snowplow', 'WashingtonPost',
]
EngineClass = create_engine_class()
messages_formatter = partial(__import__('core.messages', fromlist=['FEW_SHOTS_MESSAGES_FORMATTER']).FEW_SHOTS_MESSAGES_FORMATTER, num_shots=0)
for profile in (QWEN3_PROFILE, QWEN35_PROFILE):
    active = start_day10_runtime(profile)
    for mode in (Mode.PROMPT, Mode.GRAMMAR, Mode.REPAIR, Mode.THINKING):
        engine = EngineClass(EngineClass.Config(
            client=active['mode_clients']['own'][mode], model=profile.model,
            tokenizer=active['renderer'].tokenizer, mode=mode,
        ))
        for split in JSONSCHEMA_SPLITS:
            rows = list(load_dataset('epfl-dlab/JSONSchemaBench', split, split='test'))
            chosen = deterministic_subset(rows, 25, seed=10)
            report = run_selected_generation(engine, split, chosen, messages_formatter)
            atomic_write_json(
                DAY10_RECORDS / f'jsonschema-{profile.name}-{mode.value}-{split}.json',
                report,
            )
        engine.close()
print('JSONSchemaBench generation complete')


In [ ]:
from toolsmith.prompting import with_response_schema
from toolsmith.sampling.controller import FixedMaskController
from toolsmith.sampling.hf import generate_constrained, generate_unconstrained

stop_day10_runtime()
sampler_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B', token=HF_TOKEN, use_fast=False)
sampler_config = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B', token=HF_TOKEN)
sampler_vocabulary = Vocabulary.from_qwen_tokenizer(
    sampler_tokenizer,
    model_vocabulary_size=sampler_config.vocab_size,
    eos_token_ids=frozenset({sampler_config.eos_token_id}),
)
sampler_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen3-0.6B', token=HF_TOKEN, torch_dtype=torch.float16,
).to('cuda').eval()
hf_renderer = ChatPromptRenderer(sampler_tokenizer)
for split in JSONSCHEMA_SPLITS:
    rows = list(load_dataset('epfl-dlab/JSONSchemaBench', split, split='test'))
    for item in deterministic_subset(rows, 25, seed=10):
        schema = item['json_schema']
        if isinstance(schema, str):
            schema = json.loads(schema)
        messages = messages_formatter(split, schema)
        prompt = hf_renderer.render(with_response_schema(messages, {
            'type': 'json_schema',
            'json_schema': {'name': 'jsonschemabench', 'schema': schema, 'strict': True},
        }), enable_thinking=False)
        compiled = GBNFCompiler().compile_schema(schema)
        for mode in ('prompt', 'fixed'):
            controller = FixedMaskController(
                sampler_vocabulary, PushdownAutomaton.from_compiled(compiled)
            ) if mode == 'fixed' else None
            generate = generate_constrained if controller else generate_unconstrained
            arguments = (controller,) if controller else ()
            result = generate(
                sampler_model, sampler_tokenizer, sampler_vocabulary, prompt,
                *arguments, max_new_tokens=1024, seed=0,
            )
            atomic_write_json(
                DAY10_RECORDS / f'hf-jsonschema-{split}-{item["unique_id"]}-{mode}.json',
                {'text': result.text, 'token_ids': result.token_ids, 'stop_reason': result.stop_reason},
            )
print('HF Qwen3-0.6B JSONSchemaBench generation complete')


In [ ]:
from toolsmith.evaluation.maskbench_engine import ToolsmithMaskEngine
from toolsmith.evaluation.maskbench_runner import run_maskbench_files, timing_summary

stop_day10_runtime()
for variable in ('sampler_model', 'sampler_tokenizer', 'sampler_config'):
    globals().pop(variable, None)
if 'torch' in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()
slow_llama = AutoTokenizer.from_pretrained(
    DAY10_LOCK['maskbench_tokenizer'], token=HF_TOKEN, use_fast=False,
)
fast_llama = AutoTokenizer.from_pretrained(
    DAY10_LOCK['maskbench_tokenizer'], token=HF_TOKEN, use_fast=True,
)
for text in ('ASCII text', '  whitespace\nnext', '雪😀 Zürich', '{"nested":[1,true,null]}'):
    slow_ids = slow_llama(text, add_special_tokens=False)['input_ids']
    fast_ids = fast_llama(text, add_special_tokens=False)['input_ids']
    if slow_ids != fast_ids:
        raise AssertionError('Fast/slow Llama token IDs differ')
    custom_vocabulary = Vocabulary.from_llama_tokenizer(slow_llama)
    if custom_vocabulary.reconstruct_bytes(slow_ids) != text.encode('utf-8'):
        raise AssertionError('Llama raw-byte reconstruction failed')
custom_engine = ToolsmithMaskEngine(cache_size=4096)
custom_engine.tokenizer = slow_llama
custom_engine.tokenizer_model_id = DAY10_LOCK['maskbench_tokenizer']
custom_engine.init()
custom_results = run_maskbench_files(
    custom_engine, day10_maskbench_files, DAY10_RECORDS / 'maskbench/toolsmith',
)
atomic_write_json(DAY10_RECORDS / 'maskbench/toolsmith-summary.json', timing_summary(custom_results))
for flag in ('--llg', '--xgr', '--llamacpp', '--outlines', '--grid'):
    output = DAY10_RECORDS / f'maskbench/{flag[2:]}'
    output.mkdir(parents=True, exist_ok=True)
    run_checked([
        PEER_PYTHON, '-m', 'maskbench.runner', flag,
        '--num-threads', '1', '--tokenizer', DAY10_LOCK['maskbench_tokenizer'],
        '--output', output, *day10_maskbench_files,
    ], env=PEER_ENV)
print('MaskBench custom and peer runs complete')


In [ ]:
from collections import Counter
from toolsmith.evaluation.maskbench_runner import timing_summary
from toolsmith.evaluation.report import markdown_summary, summarize_by_group

own_flat = [
    result
    for profile_results in DAY10_OWN_RESULTS.values()
    for mode_results in profile_results.values()
    for result in mode_results
]
own_summary = summarize_by_group(own_flat)
summary = {
    'own': own_summary,
    'bfcl_score_directory': str(DAY10_RECORDS / 'bfcl-score'),
    'maskbench_toolsmith': timing_summary(custom_results),
    'maskbench_peer_directories': sorted(str(path) for path in (DAY10_RECORDS / 'maskbench').iterdir()),
    'lock': DAY10_LOCK,
}
atomic_write_json(DAY10_RECORDS / 'summary.json', summary)
(DAY10_RECORDS / 'RESULTS.md').write_text(
    markdown_summary(own_summary, title='Toolsmith Day 10 evaluation')
    + '\nOfficial BFCL scores: `bfcl-score/`. JSONSchemaBench outputs: `jsonschema-*`. MaskBench: `maskbench/`.\n'
)
failure_examples = []
for result in own_flat:
    if result.status not in {'success', 'repaired'} or result.semantic_valid is False:
        failure_examples.append(result.as_dict())
failure_recorder = JsonlRecorder(DAY10_RECORDS / 'failure-examples.jsonl', secrets=(HF_TOKEN, LOCAL_KEY))
for failure in failure_examples:
    failure_recorder.append(failure)
atomic_write_json(DAY10_RECORDS / 'manifest.json', {
    'dataset': day10_dataset_summary,
    'lock': DAY10_LOCK,
    'bfcl_selected_ids': bfcl_eligible,
    'maskbench_schema_count': len(day10_maskbench_files),
})
print(f'Day 10 report: {DAY10_RECORDS / "RESULTS.md"}')


## Steadyhand part 1: bare episode loop

Steadyhand is the agent loop that runs on top of the shim. It sends the conversation and the ten mock-store tool schemas through `ToolsmithService.chat` with `tool_choice="auto"`, grammars on, and `enable_thinking=false`. It then runs the native tool calls that come back against an in-memory store that belongs to that episode, and stops on a text answer, an error, or `max_steps`. Handlers run in Steadyhand after `chat()` returns. The FastAPI routes never call them.

This cell starts no model. It runs two short scripted episodes at the same time against a llama-server that is **already running**, and records whether their `chat()` calls overlapped in wall-clock time. If `TOOLSMITH_BASE_URL` is unset or not on localhost, the cell records why it skipped and does nothing else. To point it at the Day 10 server (which runs with `parallel=4`), run:

```python
os.environ['TOOLSMITH_BASE_URL'] = DAY10_ACTIVE['server'].base_url
os.environ['STEADYHAND_PROFILE'] = DAY10_ACTIVE['profile'].name  # 'qwen3' or 'qwen35'
```

The API key is `LOCAL_KEY` from Colab Secrets. Journals go to `/content/steadyhand/<episode_id>.jsonl` on local SSD, with `<episode_id>.manifest.json` beside each one, so Drive FUSE latency stays off the worker. A background copier moves each finished episode to `MyDrive/toolsmith/records/steadyhand/`, and the cell waits for it to drain. Finished journals replay on CPU with `steadyhand.replay.replay_episode`.

In [ ]:
import concurrent.futures
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

from steadyhand import (
    Budgets,
    LiveConfigError,
    LiveSettings,
    SteadyhandClient,
    build_llama_client,
    build_registry,
    initial_messages,
    load_fixture,
    worker_service,
)
from steadyhand.journal import JournalCopier
from steadyhand.replay import run_claimed_episode
from toolsmith.prompting import ChatPromptRenderer
from toolsmith.protocols import QWEN3_PROFILE, QWEN35_PROFILE
from toolsmith.recording import JsonlRecorder

STEADYHAND_LOCAL_DIR = Path(os.environ.get('STEADYHAND_LOCAL_DIR', '/content/steadyhand'))
STEADYHAND_DRIVE_DIR = Path(DRIVE_RECORDS) / 'steadyhand'
STEADYHAND_SERVER_PARALLEL = int(os.environ.get('TOOLSMITH_PARALLEL', '4'))
STEADYHAND_MAX_STEPS = 4
STEADYHAND_MAX_TOKENS = 512
STEADYHAND_SEED = 0
STEADYHAND_SMOKE_TASKS = (
    ('order-status', 'What is the current status of order ord_1001?'),
    ('payment-check', 'How was order ord_1003 paid for, and has that payment been captured?'),
)
steadyhand_secrets = (HF_TOKEN, LOCAL_KEY)
steadyhand_log = JsonlRecorder(STEADYHAND_DRIVE_DIR / 'part1-smoke.jsonl', secrets=steadyhand_secrets)
steadyhand_profiles = {'qwen3': QWEN3_PROFILE, 'qwen35': QWEN35_PROFILE}
steadyhand_profile_name = os.environ.get('STEADYHAND_PROFILE', 'qwen3')

steadyhand_skip_reason = None
try:
    steadyhand_settings = LiveSettings.from_env(api_key=LOCAL_KEY)
except LiveConfigError as steadyhand_error:
    steadyhand_settings, steadyhand_skip_reason = None, str(steadyhand_error)
if steadyhand_skip_reason is None and steadyhand_profile_name not in steadyhand_profiles:
    steadyhand_skip_reason = f'STEADYHAND_PROFILE must be one of {sorted(steadyhand_profiles)}'

if steadyhand_skip_reason is not None:
    steadyhand_record = steadyhand_log.append({
        'event': 'steadyhand_part1_smoke',
        'status': 'skipped',
        'reason': steadyhand_skip_reason,
    })
    print(f'Steadyhand live smoke skipped: {steadyhand_skip_reason}')
else:
    steadyhand_profile = steadyhand_profiles[steadyhand_profile_name]
    steadyhand_active = globals().get('DAY10_ACTIVE') or {}
    steadyhand_renderer = None
    if steadyhand_active.get('profile') == steadyhand_profile:
        steadyhand_renderer = steadyhand_active.get('renderer')
    if steadyhand_renderer is None:
        steadyhand_renderer = globals().get(
            'renderer' if steadyhand_profile == QWEN3_PROFILE else 'qwen35_renderer'
        )
    if steadyhand_renderer is None:
        steadyhand_renderer = ChatPromptRenderer.from_pretrained(
            steadyhand_profile.tokenizer, token=HF_TOKEN,
        )
    steadyhand_registry = build_registry()
    steadyhand_fixture = load_fixture()
    steadyhand_connections = max(STEADYHAND_SERVER_PARALLEL, len(STEADYHAND_SMOKE_TASKS))
    steadyhand_llama = build_llama_client(
        steadyhand_settings, connections=steadyhand_connections, timeout=600.0,
    )

    class TimedSteadyhandClient(SteadyhandClient):
        """Records perf_counter intervals around each chat() call."""

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.intervals = []

        def complete(self, messages, tools, *, tool_choice='auto'):
            started = time.perf_counter()
            try:
                return super().complete(messages, tools, tool_choice=tool_choice)
            finally:
                self.intervals.append((started, time.perf_counter()))

    # A fresh label per cell run, so re-running starts new episodes instead of
    # finding the earlier manifests completed and skipping.
    steadyhand_run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    steadyhand_copier = JournalCopier(STEADYHAND_DRIVE_DIR)

    def run_steadyhand_smoke_episode(label, question):
        # One ToolsmithService per worker; all of them share steadyhand_llama.
        timed_client = TimedSteadyhandClient(
            worker_service(steadyhand_llama, steadyhand_renderer, steadyhand_profile),
            max_tokens=STEADYHAND_MAX_TOKENS,
            seed=STEADYHAND_SEED,
        )
        result = run_claimed_episode(
            STEADYHAND_LOCAL_DIR,
            timed_client,
            steadyhand_registry,
            steadyhand_fixture.clone(),
            initial_messages(question),
            budgets=Budgets(max_steps=STEADYHAND_MAX_STEPS),
            secrets=steadyhand_secrets,
            run_label=f'smoke-{label}-{steadyhand_run_stamp}',
        )
        if result is None:
            return {'label': label, 'status': 'skipped', 'intervals': timed_client.intervals}
        steadyhand_copier.submit(result.journal_path)
        return {
            'label': label,
            'episode_id': result.episode_id,
            'status': result.status.value,
            'steps': result.steps,
            'error_code': result.error_code,
            'final_content': result.final_content,
            'prompt_tokens': result.prompt_tokens,
            'completion_tokens': result.completion_tokens,
            'journal': str(result.journal_path),
            'drive_copy': str(STEADYHAND_DRIVE_DIR / result.journal_path.name),
            'intervals': timed_client.intervals,
        }

    def overlap_seconds(first, second):
        return sum(
            max(0.0, min(first_end, second_end) - max(first_start, second_start))
            for first_start, first_end in first
            for second_start, second_end in second
        )

    steadyhand_started = time.perf_counter()
    try:
        with concurrent.futures.ThreadPoolExecutor(
            max_workers=len(STEADYHAND_SMOKE_TASKS), thread_name_prefix='steadyhand',
        ) as steadyhand_pool:
            steadyhand_futures = [
                steadyhand_pool.submit(run_steadyhand_smoke_episode, label, question)
                for label, question in STEADYHAND_SMOKE_TASKS
            ]
            steadyhand_runs = [future.result() for future in steadyhand_futures]
    finally:
        steadyhand_llama.close()
        steadyhand_copier.close()  # waits until every finished journal is on Drive
    steadyhand_overlap = overlap_seconds(
        steadyhand_runs[0]['intervals'], steadyhand_runs[1]['intervals'],
    )
    for steadyhand_run in steadyhand_runs:
        steadyhand_run['intervals'] = [
            [round(start - steadyhand_started, 4), round(end - steadyhand_started, 4)]
            for start, end in steadyhand_run['intervals']
        ]
    steadyhand_record = steadyhand_log.append({
        'event': 'steadyhand_part1_smoke',
        'status': 'ran',
        'profile': steadyhand_profile.name,
        'model': steadyhand_profile.model,
        'client_connections': steadyhand_connections,
        'episodes': steadyhand_runs,
        'completions_overlapped': steadyhand_overlap > 0,
        'overlap_seconds': round(steadyhand_overlap, 4),
        'copy_errors': steadyhand_copier.errors,
    })
    print(json.dumps(steadyhand_record, indent=2, ensure_ascii=False))

## Steadyhand part 7: evaluation sweep

The loop from parts 1–6 is scored on 60 mock-store tasks (`evaluation/data/steadyhand_tasks_v1.jsonl`). Each task's gold is a diff of the final store, not the wording of the answer. Each task runs clean and under every fault variant it declares. The fault seed is the task seed, and policies share the same pinned faults.

Run these cells in order, one model at a time:

1. **CPU gate.** Runs the dataset validator, all six policies on the scripted model, and the golden replays. It needs no GPU, and can run while llama-server loads. It must pass before any scored episode.
2. **Server.** Starts the only llama-server, from `STEADYHAND_PHASE` (`qwen35/wide`, `qwen35/tall`, `qwen3/wide`, `qwen3/tall`). Context and slots are set as a pair: `per_slot_context = context_size // parallel`. The probe drops one slot at a time while VRAM stays under 1 GB free. It then exports `TOOLSMITH_BASE_URL`.
3. **Smoke.** Runs `parallel` scripted-length episodes at once and prints single-stream versus aggregate tokens/sec. It accepts the shape only when every slot is busy.
4. **Sweep.** Runs `workers == parallel` and resumes from the sweep manifest. Journals go to local disk, finished episodes are copied to `MyDrive/toolsmith/records/steadyhand/`, and `RESULTS.md` is rendered from them.
5. **Stop.** Frees the GPU before the next phase or model.

Choose the grid with `STEADYHAND_POLICIES` (default `typed_hint,escalate`, the first live table), `STEADYHAND_PROMPTS` (`plain`, optionally `failure_aware`), and `STEADYHAND_VARIANTS`. The presets are capacity guesses for the probe to confirm, not measurements. Infrastructure aborts stay in the log and never count as task results.

In [ ]:
import json
import os
import tempfile
from pathlib import Path

from steadyhand.eval.suite import load_tasks, run_cpu_suite, validate_tasks
from steadyhand.recovery import STRATEGIES
from steadyhand.replay import replay_many
from toolsmith.recording import JsonlRecorder

STEADYHAND_GOLDEN_DIR = Path(PROJECT_DIR) / 'tests' / 'fixtures' / 'steadyhand_episodes'
steadyhand_tasks = load_tasks()
steadyhand_dataset = validate_tasks(steadyhand_tasks)
steadyhand_gate_limit = int(os.environ.get('STEADYHAND_CPU_GATE_TASKS') or 0)
steadyhand_gate_tasks = steadyhand_tasks[:steadyhand_gate_limit] if steadyhand_gate_limit else steadyhand_tasks
steadyhand_gate_workers = int(os.environ.get('STEADYHAND_CPU_GATE_WORKERS') or min(os.cpu_count() or 1, 8))
steadyhand_gate_dir = Path(tempfile.mkdtemp(prefix='steadyhand-cpu-gate-'))

# Every policy on the scripted model, then a blind-retry pass on the lost-ack slice.
steadyhand_cpu_scores = run_cpu_suite(
    steadyhand_gate_tasks, sorted(STRATEGIES),
    directory=steadyhand_gate_dir / 'follow-hints', workers=steadyhand_gate_workers,
)
steadyhand_blind_scores = run_cpu_suite(
    steadyhand_gate_tasks, ['raw_error', 'retry_safe', 'idempotent_verify', 'escalate'],
    directory=steadyhand_gate_dir / 'blind-retry', behavior='blind_retry',
    variants=['lost_acknowledgment'], workers=steadyhand_gate_workers,
)
steadyhand_golden = replay_many(sorted(STEADYHAND_GOLDEN_DIR.glob('ep_*.jsonl')), workers=steadyhand_gate_workers)

steadyhand_gate_problems = [
    f'{score.task_id} clean failed under {score.policy}'
    for score in steadyhand_cpu_scores
    if score.variant == 'clean' and score.policy in ('typed_hint', 'escalate') and not score.task_success
]
steadyhand_gate_problems += [
    f'{score.task_id} duplicated an effect under {score.policy}'
    for score in steadyhand_blind_scores
    if score.policy in ('idempotent_verify', 'escalate') and score.duplicate_side_effects
]
steadyhand_gate_problems += [f'golden replay failed: {outcome.error}' for outcome in steadyhand_golden if not outcome.ok]
steadyhand_gate_record = {
    'event': 'steadyhand_cpu_gate',
    'dataset': steadyhand_dataset,
    'tasks_run': len(steadyhand_gate_tasks),
    'episodes': len(steadyhand_cpu_scores) + len(steadyhand_blind_scores),
    'golden_replays': len(steadyhand_golden),
    'problems': steadyhand_gate_problems,
    'passed': not steadyhand_gate_problems,
}
JsonlRecorder(
    Path(DRIVE_RECORDS) / 'steadyhand' / 'cpu-gate.jsonl', secrets=(HF_TOKEN, LOCAL_KEY),
).append(steadyhand_gate_record)
print(json.dumps({key: value for key, value in steadyhand_gate_record.items() if key != 'dataset'}, indent=2))
if steadyhand_gate_problems:
    raise RuntimeError('Steadyhand CPU gate failed; fix it before any scored GPU episode')

In [ ]:
import subprocess

from steadyhand import LiveSettings, build_llama_client
from steadyhand.eval.serving import needs_smaller_shape, parse_nvidia_smi, preset_for, server_extra_args
from toolsmith.backends import LlamaCppClient, LlamaCppServer
from toolsmith.prompting import ChatPromptRenderer
from toolsmith.protocols import QWEN3_PROFILE, QWEN35_PROFILE

def stop_steadyhand_runtime():
    global STEADYHAND_ACTIVE
    active = globals().get('STEADYHAND_ACTIVE') or {}
    if active.get('client') is not None:
        active['client'].close()
    if active.get('server') is not None:
        active['server'].stop(timeout=30)
    STEADYHAND_ACTIVE = {}
    os.environ.pop('TOOLSMITH_BASE_URL', None)

def query_gpu():
    output = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.used,utilization.gpu',
         '--format=csv,noheader,nounits'],
        check=True, capture_output=True, text=True,
    ).stdout
    return parse_nvidia_smi(output)[0]

missing = [name for name in ('SERVER_BIN', 'LOCAL_MODEL', 'LOCAL_BUILD', 'LOCAL_KEY', 'server_environment', 'first_free_port')
           if name not in globals()]
if missing:
    raise RuntimeError(f'Run the build, download, and start-server cells first; missing: {missing}')

steadyhand_profile_name, steadyhand_shape = os.environ.get('STEADYHAND_PHASE', 'qwen35/wide').split('/')
steadyhand_profile = {'qwen3': QWEN3_PROFILE, 'qwen35': QWEN35_PROFILE}[steadyhand_profile_name]
if steadyhand_profile_name == 'qwen35' and 'QWEN35_SERVER_BIN' not in globals():
    raise RuntimeError('Run the Qwen3.5 runtime cell first, or set STEADYHAND_PHASE=qwen3/wide')

# One resident GGUF: stop every other llama-server before loading this one.
if 'stop_day10_runtime' in globals():
    stop_day10_runtime()
if 'stop_listening_llama_servers' in globals():
    stop_listening_llama_servers()
stop_steadyhand_runtime()

if steadyhand_profile_name == 'qwen3':
    steadyhand_binary, steadyhand_model, steadyhand_build = SERVER_BIN, LOCAL_MODEL, LOCAL_BUILD
else:
    steadyhand_binary, steadyhand_model, steadyhand_build = QWEN35_SERVER_BIN, QWEN35_LOCAL_MODEL, QWEN35_LOCAL_BUILD
steadyhand_env = server_environment(steadyhand_build)
os.environ['LD_LIBRARY_PATH'] = steadyhand_env['LD_LIBRARY_PATH']
steadyhand_help = subprocess.run(
    [str(steadyhand_binary), '--help'], capture_output=True, text=True, env=steadyhand_env,
).stdout
steadyhand_extra_args = server_extra_args(steadyhand_help, cpu_count=os.cpu_count() or 1)
steadyhand_gpu = query_gpu()
steadyhand_preset = preset_for(steadyhand_gpu['name'], steadyhand_profile_name, steadyhand_shape)
steadyhand_port = first_free_port('127.0.0.1', 8090)
steadyhand_log = Path('/content/steadyhand-server.log')

while True:
    steadyhand_server = LlamaCppServer(
        executable=steadyhand_binary, model=steadyhand_model, api_key=LOCAL_KEY,
        log_path=steadyhand_log, port=steadyhand_port, context_size=steadyhand_preset.context_size,
        gpu_layers=99, parallel=steadyhand_preset.parallel, extra_args=steadyhand_extra_args,
    )
    steadyhand_probe = LlamaCppClient(steadyhand_server.base_url, LOCAL_KEY, timeout=30.0)
    steadyhand_reason = None
    try:
        steadyhand_server.start()
        steadyhand_server.wait_until_healthy(steadyhand_probe, timeout=900, poll_interval=2)
        steadyhand_gpu = query_gpu()
        if needs_smaller_shape(steadyhand_gpu):
            steadyhand_reason = 'less than 1 GB of VRAM free after load'
    except Exception as steadyhand_error:
        steadyhand_reason = f'{type(steadyhand_error).__name__}: {steadyhand_error}'
    finally:
        steadyhand_probe.close()
    if steadyhand_reason is None:
        break
    steadyhand_server.stop(timeout=30)
    if steadyhand_preset.parallel <= 1:
        raise RuntimeError(f'{steadyhand_preset.name} does not fit even with one slot: {steadyhand_reason}')
    print(f'{steadyhand_preset.name} at parallel={steadyhand_preset.parallel}: {steadyhand_reason}; one slot fewer')
    steadyhand_preset = steadyhand_preset.shrink()

os.environ['TOOLSMITH_BASE_URL'] = steadyhand_server.base_url
steadyhand_renderer = globals().get('renderer' if steadyhand_profile_name == 'qwen3' else 'qwen35_renderer')
if steadyhand_renderer is None:
    steadyhand_renderer = ChatPromptRenderer.from_pretrained(steadyhand_profile.tokenizer, token=HF_TOKEN)
STEADYHAND_ACTIVE = {
    'server': steadyhand_server,
    'preset': steadyhand_preset,
    'profile': steadyhand_profile,
    'renderer': steadyhand_renderer,
    'gpu': steadyhand_gpu,
    'extra_args': steadyhand_extra_args,
    # The pool admits a little more than parallel so verify or backoff never queues a slot.
    'client': build_llama_client(
        LiveSettings.from_env(api_key=LOCAL_KEY),
        connections=steadyhand_preset.parallel + 2, timeout=900.0,
    ),
}
steadyhand_slot_lines = [line for line in steadyhand_log.read_text(errors='replace').splitlines() if 'n_ctx' in line][:4]
print(json.dumps({**steadyhand_preset.as_dict(), 'gpu': steadyhand_gpu, 'extra_args': steadyhand_extra_args}, indent=2))
print('Server log context lines (confirm the per-slot split):', *steadyhand_slot_lines, sep='\n  ')

In [ ]:
import concurrent.futures
import threading
from datetime import datetime, timezone

import httpx

from steadyhand import SteadyhandClient, worker_service
from steadyhand.eval.serving import slots_busy, tokens_per_second
from steadyhand.eval.suite import load_tasks, run_task

if not globals().get('STEADYHAND_ACTIVE'):
    raise RuntimeError('Run the Steadyhand server cell first')
steadyhand_active = STEADYHAND_ACTIVE
steadyhand_parallel = steadyhand_active['preset'].parallel
steadyhand_smoke_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
steadyhand_smoke_dir = Path('/content/steadyhand/smoke') / steadyhand_smoke_stamp
steadyhand_smoke_tasks = [task for task in load_tasks() if task.category == 'single_step'][:steadyhand_parallel]

def steadyhand_live_client(task):
    return SteadyhandClient(
        worker_service(steadyhand_active['client'], steadyhand_active['renderer'], steadyhand_active['profile']),
        max_tokens=int(os.environ.get('STEADYHAND_MAX_TOKENS', '512')),
        seed=task.seed,
        slot_context=steadyhand_active['preset'].per_slot_context,
    )

def steadyhand_smoke_episode(task, label):
    return run_task(
        task, 'clean', policy='typed_hint', directory=steadyhand_smoke_dir / label,
        client=steadyhand_live_client(task), run_label=f'smoke-{label}-{steadyhand_smoke_stamp}',
    )

steadyhand_started = time.perf_counter()
steadyhand_single = steadyhand_smoke_episode(steadyhand_smoke_tasks[0], 'single')
steadyhand_single_tps = tokens_per_second(steadyhand_single.completion_tokens, time.perf_counter() - steadyhand_started)

steadyhand_samples, steadyhand_memory = [], []
steadyhand_sampling = threading.Event()

def steadyhand_sample_slots():
    headers = {'Authorization': f'Bearer {LOCAL_KEY}'}
    with httpx.Client(base_url=steadyhand_active['server'].base_url, headers=headers, timeout=5.0) as http:
        while not steadyhand_sampling.is_set():
            try:
                response = http.get('/slots')
                if response.status_code == 200:
                    steadyhand_samples.append(slots_busy(response.json()))
                steadyhand_memory.append(query_gpu())
            except (httpx.HTTPError, ValueError, subprocess.SubprocessError):
                pass
            time.sleep(0.25)

steadyhand_sampler = threading.Thread(target=steadyhand_sample_slots, daemon=True)
steadyhand_sampler.start()
steadyhand_started = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=steadyhand_parallel) as steadyhand_pool:
    steadyhand_smoke_scores = list(steadyhand_pool.map(
        lambda task: steadyhand_smoke_episode(task, 'aggregate'), steadyhand_smoke_tasks,
    ))
steadyhand_elapsed = time.perf_counter() - steadyhand_started
steadyhand_sampling.set()
steadyhand_sampler.join(timeout=5)

steadyhand_busy = max((busy for busy, _ in steadyhand_samples), default=None)
STEADYHAND_SMOKE = {
    'gpu': steadyhand_active['gpu']['name'],
    'preset': steadyhand_active['preset'].name,
    'parallel': steadyhand_parallel,
    'per_slot_context': steadyhand_active['preset'].per_slot_context,
    'single_stream_tps': steadyhand_single_tps,
    'aggregate_tps': tokens_per_second(sum(s.completion_tokens for s in steadyhand_smoke_scores), steadyhand_elapsed),
    'slots_busy': f'{steadyhand_busy}/{steadyhand_parallel}' if steadyhand_busy is not None else 'unknown',
    'peak_memory_mib': max((row['memory_used_mib'] for row in steadyhand_memory), default=None),
    'peak_gpu_utilization': max((row['utilization_gpu'] for row in steadyhand_memory), default=None),
    'server_alive': steadyhand_active['server'].is_running,
}
print(json.dumps(STEADYHAND_SMOKE, indent=2))
steadyhand_accepted = steadyhand_busy == steadyhand_parallel and STEADYHAND_SMOKE['server_alive']
if not steadyhand_accepted and os.environ.get('STEADYHAND_ACCEPT_SMOKE') != '1':
    raise RuntimeError(
        'Not every slot was busy during the smoke (or /slots is unavailable). Check that the client issues '
        'concurrent complete() calls, or set STEADYHAND_ACCEPT_SMOKE=1 after checking by hand.'
    )

In [ ]:
from steadyhand import SteadyhandClient, worker_service
from steadyhand.eval.report import render_report, write_report
from steadyhand.eval.suite import SweepSpec, dataset_sha256, load_tasks, run_sweep
from steadyhand.journal import JournalCopier
from toolsmith.artifacts import copy_file_atomic

if not globals().get('STEADYHAND_ACTIVE') or not globals().get('STEADYHAND_SMOKE'):
    raise RuntimeError('Run the Steadyhand server and smoke cells first')
steadyhand_active = STEADYHAND_ACTIVE
steadyhand_preset = steadyhand_active['preset']
steadyhand_tasks = load_tasks()
steadyhand_policies = tuple(os.environ.get('STEADYHAND_POLICIES', 'typed_hint,escalate').split(','))
steadyhand_prompts = tuple(os.environ.get('STEADYHAND_PROMPTS', 'plain').split(','))
steadyhand_variants = tuple(os.environ.get(
    'STEADYHAND_VARIANTS', 'clean,lost_acknowledgment,explicit_transient,implicit_transient',
).split(','))
steadyhand_max_tokens = int(os.environ.get('STEADYHAND_MAX_TOKENS', '512'))
steadyhand_local = Path('/content/steadyhand')
steadyhand_drive = Path(DRIVE_RECORDS) / 'steadyhand'
steadyhand_spec = SweepSpec(
    sweep_id=f"{steadyhand_preset.profile}-{steadyhand_preset.shape}-{steadyhand_preset.gpu}-"
             f"{'-'.join(steadyhand_policies)}-{'-'.join(steadyhand_prompts)}-{dataset_sha256(steadyhand_tasks)[:8]}",
    task_ids=tuple(task.task_id for task in steadyhand_tasks),
    policies=steadyhand_policies,
    prompt_variants=steadyhand_prompts,
    variants=steadyhand_variants,
    profile=steadyhand_preset.profile,
    dataset_sha256=dataset_sha256(steadyhand_tasks),
    model={
        'model': steadyhand_active['profile'].model, 'use_grammars': True, 'enable_thinking': False,
        'temperature': 0.0, 'max_tokens': steadyhand_max_tokens,
    },
    server={
        **steadyhand_preset.as_dict(), 'gpu': steadyhand_active['gpu']['name'],
        'kv_cache_type': 'q8_0', 'batch_size': 2048, 'ubatch_size': 512,
        'extra_args': steadyhand_active['extra_args'],
    },
)

def steadyhand_client_for(unit):
    return SteadyhandClient(
        worker_service(steadyhand_active['client'], steadyhand_active['renderer'], steadyhand_active['profile']),
        max_tokens=steadyhand_max_tokens,
        seed=unit.task.seed,
        slot_context=steadyhand_preset.per_slot_context,
    )

steadyhand_started = time.perf_counter()
with JournalCopier(steadyhand_drive / 'journals') as steadyhand_copier:
    steadyhand_scores = run_sweep(
        steadyhand_spec, steadyhand_tasks, steadyhand_client_for,
        journal_dir=steadyhand_local / 'journals', sweep_dir=steadyhand_local / 'sweeps',
        workers=steadyhand_preset.parallel, copier=steadyhand_copier,
        sleep=time.sleep, secrets=(HF_TOKEN, LOCAL_KEY),
    )
print(f'Sweep finished in {time.perf_counter() - steadyhand_started:.0f}s; copy errors: {steadyhand_copier.errors}')
for steadyhand_name in (f'{steadyhand_spec.sweep_id}.json', f'{steadyhand_spec.sweep_id}.scores.jsonl'):
    copy_file_atomic(steadyhand_local / 'sweeps' / steadyhand_name, steadyhand_drive / 'sweeps' / steadyhand_name)
steadyhand_report = render_report(steadyhand_scores, sweeps=[steadyhand_spec.as_dict()], smoke=[STEADYHAND_SMOKE])
write_report(steadyhand_local / 'RESULTS.md', steadyhand_report)
copy_file_atomic(steadyhand_local / 'RESULTS.md', steadyhand_drive / 'RESULTS.md')
print(steadyhand_report.split('## 3.')[0])

In [ ]:
if 'stop_steadyhand_runtime' in globals():
    stop_steadyhand_runtime()
print('Steadyhand llama-server stopped. Start the next phase or model, or disconnect once the Drive copies are done.')

In [ ]:
if 'stop_steadyhand_runtime' in globals():
    stop_steadyhand_runtime()
if 'stop_day10_runtime' in globals():
    stop_day10_runtime()
for variable in ('sampler_model', 'sampler_tokenizer', 'sampler_config'):
    globals().pop(variable, None)
if 'torch' in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()
for variable in ('qwen35_openai_client', 'openai_client'):
    if variable in globals():
        globals()[variable].close()
for server_variable, thread_variable in (
    ('qwen35_shim_server', 'qwen35_shim_thread'),
    ('shim_server', 'shim_thread'),
):
    if server_variable in globals():
        globals()[server_variable].should_exit = True
        if thread_variable in globals():
            globals()[thread_variable].join(timeout=10)
for variable in ('qwen35_client', 'client'):
    if variable in globals():
        globals()[variable].close()
for variable in ('qwen35_server', 'server'):
    if variable in globals():
        globals()[variable].stop(timeout=20)
print('FastAPI shims and managed llama-servers stopped. Record Colab units used, then delete or disconnect the runtime.')